In [128]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import pickle

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from datetime import datetime
from pandas.tseries.offsets import MonthEnd

input_table = 'TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU_NEW'
month_run = '2026-08-31'
os.listdir('/data/aman_singh/acuuracy_check')

['Stat Demand Forecast GT_as_on_6th_July_2026.xlsb',
 'Heuristics_all_combination_qcom_cp_apr_live.xlsx',
 'anaplan_snapshots_july.xlsx',
 'Heuristics_all_combination_ecom_mar_live.xlsx',
 'combine_model+missing_forecasts_brand_asm.ipynb',
 'MARICO LIMITED_swiggy_june.xlsx',
 'QCOM Chain PSKU OTP Output',
 'all_combination_qcom_aug_pred.csv',
 'key_check.csv',
 'Norms 202607.csv',
 'acc_framework_may_final.xlsx',
 'acc_offtakes_till_may.csv',
 'swigy_vol_chk.csv',
 'offtakes_monthly_aggregated.csv',
 'Marico Ltd._forecast_Jul 2026_to_Oct 2026.csv',
 'all_combination_ecom_backtest_pred.csv',
 'Stat Demand Forecast MT_as_on_9th_June_2026.xlsb',
 'ecom_chain_psku_offtake_to_secondary_v6_PROD.ipynb',
 'all_combination_MT_live_aug.csv',
 'seasonality.xlsx',
 'Qcom_chain_fc_psku_forecast_as_on_11th_may_2026.xlsx',
 'missing_df_gt_all.csv',
 'SOH - 01 Jul.xlsx',
 'qcom_chain_depot_psku_zepto_inc.xlsx',
 'Marico Ltd._forecast_Sep 2026_to_Dec 2026.csv',
 'stat_fva_till_may.csv',
 'MARICO LIMITE

In [129]:
base_dir = '/data/aman_singh/acuuracy_check'

In [130]:
def list_all_files_in_directory(root):
    out = []

    for path, subdirs, files in os.walk(root):
        for name in files:
            out.append(os.path.join(path, name))

    return out

In [131]:
list_all_files_in_directory(base_dir)

['/data/aman_singh/acuuracy_check/Stat Demand Forecast GT_as_on_6th_July_2026.xlsb',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_qcom_cp_apr_live.xlsx',
 '/data/aman_singh/acuuracy_check/anaplan_snapshots_july.xlsx',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_ecom_mar_live.xlsx',
 '/data/aman_singh/acuuracy_check/combine_model+missing_forecasts_brand_asm.ipynb',
 '/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_june.xlsx',
 '/data/aman_singh/acuuracy_check/all_combination_qcom_aug_pred.csv',
 '/data/aman_singh/acuuracy_check/key_check.csv',
 '/data/aman_singh/acuuracy_check/Norms 202607.csv',
 '/data/aman_singh/acuuracy_check/acc_framework_may_final.xlsx',
 '/data/aman_singh/acuuracy_check/acc_offtakes_till_may.csv',
 '/data/aman_singh/acuuracy_check/swigy_vol_chk.csv',
 '/data/aman_singh/acuuracy_check/offtakes_monthly_aggregated.csv',
 '/data/aman_singh/acuuracy_check/Marico Ltd._forecast_Jul 2026_to_Oct 2026.csv',
 '/data/aman_singh/acuuracy_

In [132]:
def discover_channel(file_path):
    # file_path = file_path.split('/')

    # if 'ECOM' in file_path:
    #     return 'ECOM'
    # elif 'QCOM' in file_path:
    #     return 'QCOM'
    # elif 'MT' in file_path:
    #     return 'MT'
    # else:
    #     return 'Channel not found'

    return 'ECOM'


In [133]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [134]:
data_query = f"""
    select * from {input_table}
    where month_date >= '2023-01-01' and run_month = '{month_run}'
        
"""

offtake_df = pd.read_sql(data_query, dev_conn)
offtake_df.head()

,MONTH_DATE,PLATFORM_NAME,PARENT_MATERIAL_CODE,VOL_IN_RUM,INDEXBPM,BRAND_CODE,IMPUTED,BIG_BILLION_DAYS,BIG_BILLION_DAYS_LAG_1,BIG_BILLION_DAYS_LAG_2,BIG_BILLION_DAYS_LEAD_1,BIG_BILLION_DAYS_LEAD_2,GREAT_INDIAN_FESTIVAL,GREAT_INDIAN_FESTIVAL_LAG_1,GREAT_INDIAN_FESTIVAL_LAG_2,GREAT_INDIAN_FESTIVAL_LEAD_1,GREAT_INDIAN_FESTIVAL_LEAD_2,RATIO_LAST_YEAR,QUARTER,RUN_MONTH
0,2026-07-31,Amazon ARIPL,718288,33.954,47.150311,SAFF GOLD,0,0,0,0,0,1,0,0,0,0,1,NaN,3,2026-08-31
1,2026-08-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,0,1,0,0,0,0,1,0,NaN,3,2026-08-31
2,2026-09-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,1,0,0,0,0,1,0,0,0,0,0.0,3,2026-08-31
3,2026-10-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,1,0,0,0,0,1,0,0,0,0.0,4,2026-08-31
4,2026-11-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,1,0,0,0,0,1,0,0,0.0,4,2026-08-31


In [135]:
offtake_df.columns = offtake_df.columns.str.lower()

In [136]:
offtake_df.duplicated(
    subset=['platform_name','parent_material_code', 'month_date']).sum()

0

In [137]:
offtake_df['brand_code'] = np.where(
    ((offtake_df['parent_material_code'] == 715096) &
    (offtake_df['brand_code'] == 'CO_SO_PCP')),
    'CO_SO_FS',
    offtake_df['brand_code']
)

In [138]:
# offtake_df = offtake_df[offtake_df['platform_name'].isin(
#     ['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'])]

In [139]:
offtake_df['key'] = offtake_df[['platform_name','parent_material_code']].astype(str).agg('_'.join, axis=1)
# offtake_df.rename(columns={'realigned_psku': 'parent_material_code'}, inplace=True)
# offtake_df.drop([ 'run_month'], axis=1, inplace=True)
offtake_df['parent_material_code'] = offtake_df['parent_material_code'].astype(int)

In [140]:
offtake_df.duplicated(subset=['key', 'month_date']).sum()

0

In [141]:
(offtake_df['key'] == offtake_df[['platform_name','parent_material_code']].astype(str).agg('_'.join, axis=1)).all()

True

In [142]:
# realigned_df.to_csv('OT_data_debug.csv', index=False)

### Collate MIL

In [143]:
base_dir

'/data/aman_singh/acuuracy_check'

In [144]:
def collate_file(file_hint, extension='.csv'):
    collated_file = pd.DataFrame()

    run_path = f'{base_dir}'
    all_files = list_all_files_in_directory(run_path)

    for file_path in all_files:
        if file_hint in file_path:
            if extension == '.csv':
                print(file_path)
                read_file = pd.read_csv(file_path)
                # read_file['channel'] = discover_channel(file_path)
                read_file['run'] = 'run'
                read_file['step'] = file_path.split('/')[3]
                read_file['file_path'] = file_path

                collated_file = pd.concat(
                    [collated_file, read_file]
                )
                del read_file

    return collated_file

In [145]:
trend_file_df = collate_file('trend_file_train_till')
prophet_file_df = collate_file('prophet_data_train_till')

/data/aman_singh/acuuracy_check/trend_file_train_till_31_Jul_2026 (7).csv
/data/aman_singh/acuuracy_check/trend_file_train_till_31_Jul_2026 (8).csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_Jul_2026 (8).csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_Jul_2026 (7).csv


In [146]:
# forecast_train_till_file_df = collate_file('forecast_train_till_')

In [147]:
# forecast_train_till_file_df

In [148]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,...,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2
0,Amazon RK_710542,2024-09-30,0.000000,0.000330,0.000165,NaN,0.000196,0.000000,0.000012,0.000006,...,2026-07-31,3.328158,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
1,Amazon RK_710542,2024-10-31,0.000221,0.000330,0.000165,NaN,0.000151,0.000008,0.000012,0.000006,...,2026-07-31,3.328158,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
2,Amazon RK_710542,2024-11-30,0.000266,0.000330,0.000165,NaN,0.000101,0.000009,0.000012,0.000006,...,2026-07-31,3.328158,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
3,Amazon RK_710542,2024-12-31,0.000000,0.000330,0.000165,NaN,0.000052,0.000000,0.000012,0.000006,...,2026-07-31,3.328158,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
4,Amazon RK_710542,2025-01-31,0.000000,0.000180,0.000165,NaN,0.000000,0.000000,0.000006,0.000006,...,2026-07-31,3.328158,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95250,Purplle_809250,2026-11-30,0.000940,0.000933,0.000917,0.000000,0.001135,0.000272,0.000270,0.000265,...,2026-07-31,0.717091,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0
95251,Purplle_809250,2026-12-31,0.001487,0.000933,0.000917,0.000330,0.000798,0.000430,0.000270,0.000265,...,2026-07-31,0.717091,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0
95252,Purplle_809250,2027-01-31,0.001487,0.000933,0.000917,0.000199,0.001010,0.000430,0.000270,0.000265,...,2026-07-31,0.717091,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0
95253,Purplle_809250,2027-02-28,0.000940,0.000933,0.000917,0.000648,0.001168,0.000272,0.000270,0.000265,...,2026-07-31,0.717091,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0


In [149]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'vol_in_rum', 'brand_code',
       'great_indian_festival', 'great_indian_festival_lag_1',
       'great_indian_festival_lag_2', 'great_indian_festival_lead_1',
       'great_indian_festival_lead_2', 'ratio_last_year', 'quarter',
       'qtr_ind_rate', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'big_billion_days', 'big_billion_days_lag_1',
       'big_billion_days_lag_2', 'big_billion_days_lead_1',
       'big_billion_days_lead_2'],
      dtype='object')

In [150]:
trend_file_df['platform_name'].unique()

array(['Amazon RK', 'Big Basket', 'Flipkart Grocery', 'Flipkart National',
       'Meesho', 'Myntra', 'Nykaa', 'Purplle'], dtype=object)

In [151]:
trend_file_df['month_date'] = pd.to_datetime(trend_file_df['month_date'])
prophet_file_df['month_date'] = pd.to_datetime(prophet_file_df['month_date'])

trend_file_df['train_till'] = pd.to_datetime(trend_file_df['train_till'])
prophet_file_df['train_till'] = pd.to_datetime(prophet_file_df['train_till'])

trend_file_df['run_month'] = pd.to_datetime(trend_file_df['train_till'] + MonthEnd(1))
prophet_file_df['run_month'] = pd.to_datetime(prophet_file_df['train_till'] + MonthEnd(1))

In [152]:
trend_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum(), \
prophet_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum()

(0, 0)

In [153]:
mappings = {}

for run_month in trend_file_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-08-31 00:00:00'): {Timestamp('2026-08-31 00:00:00'): 'M',
  Timestamp('2026-09-30 00:00:00'): 'M+1',
  Timestamp('2026-10-31 00:00:00'): 'M+2',
  Timestamp('2026-11-30 00:00:00'): 'M+3',
  Timestamp('2026-12-31 00:00:00'): 'M+4',
  Timestamp('2027-01-31 00:00:00'): 'M+5',
  Timestamp('2027-02-28 00:00:00'): 'M+6',
  Timestamp('2027-03-31 00:00:00'): 'M+7',
  Timestamp('2027-04-30 00:00:00'): 'M+8'}}

In [154]:
trend_file_df['M month'] = trend_file_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

In [155]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,...,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month
0,Amazon RK_710542,2024-09-30,0.000000,0.000330,0.000165,NaN,0.000196,0.000000,0.000012,0.000006,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None
1,Amazon RK_710542,2024-10-31,0.000221,0.000330,0.000165,NaN,0.000151,0.000008,0.000012,0.000006,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None
2,Amazon RK_710542,2024-11-30,0.000266,0.000330,0.000165,NaN,0.000101,0.000009,0.000012,0.000006,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None
3,Amazon RK_710542,2024-12-31,0.000000,0.000330,0.000165,NaN,0.000052,0.000000,0.000012,0.000006,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None
4,Amazon RK_710542,2025-01-31,0.000000,0.000180,0.000165,NaN,0.000000,0.000000,0.000006,0.000006,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95250,Purplle_809250,2026-11-30,0.000940,0.000933,0.000917,0.000000,0.001135,0.000272,0.000270,0.000265,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-08-31,M+3
95251,Purplle_809250,2026-12-31,0.001487,0.000933,0.000917,0.000330,0.000798,0.000430,0.000270,0.000265,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,M+4
95252,Purplle_809250,2027-01-31,0.001487,0.000933,0.000917,0.000199,0.001010,0.000430,0.000270,0.000265,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,M+5
95253,Purplle_809250,2027-02-28,0.000940,0.000933,0.000917,0.000648,0.001168,0.000272,0.000270,0.000265,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,M+6


In [156]:
trend_file_df[trend_file_df['M month'].notna()]

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,...,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month
23,Amazon RK_710542,2026-08-31,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.00000,0.000000,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,M
24,Amazon RK_710542,2026-09-30,0.000000,0.000000,0.000000,NaN,0.000045,0.000000,0.00000,0.000000,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,M+1
25,Amazon RK_710542,2026-10-31,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.00000,0.000000,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,M+2
26,Amazon RK_710542,2026-11-30,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.00000,0.000000,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,M+3
27,Amazon RK_710542,2026-12-31,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.00000,0.000000,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,M+4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95250,Purplle_809250,2026-11-30,0.000940,0.000933,0.000917,0.000000,0.001135,0.000272,0.00027,0.000265,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-08-31,M+3
95251,Purplle_809250,2026-12-31,0.001487,0.000933,0.000917,0.000330,0.000798,0.000430,0.00027,0.000265,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,M+4
95252,Purplle_809250,2027-01-31,0.001487,0.000933,0.000917,0.000199,0.001010,0.000430,0.00027,0.000265,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,M+5
95253,Purplle_809250,2027-02-28,0.000940,0.000933,0.000917,0.000648,0.001168,0.000272,0.00027,0.000265,...,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,M+6


In [157]:
trend_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-08-31,2026-07-31


In [158]:
prophet_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-08-31,2026-07-31


In [159]:
trend_file_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7'],
      dtype=object)

In [160]:
portfolio_query = """SELECT DISTINCT
    MATERIAL_GROUP_CODE AS BRAND_CODE,
    SAP_PORTFOLIO_NAME AS PORTFOLIO
FROM MST_MATERIAL
WHERE COMPANY_CODE = 'MIL'
  AND LATEST_RECORD_IND = '1'
  AND MATERIAL_GROUP_CODE IS NOT NULL
  AND SAP_PORTFOLIO_NAME IS NOT NULL
ORDER BY SAP_PORTFOLIO_NAME, MATERIAL_GROUP_CODE;"""
brand_md_df = pd.read_sql(portfolio_query, prod_conn)

brand_md_df.columns = brand_md_df.columns.str.lower()
brand_md_df

,brand_code,portfolio
0,BD_BDOL_M,BEARDO
1,BD_HRWX_M,BEARDO
2,BRD_BDCOL,BEARDO
3,BRD_BDOIL,BEARDO
4,BRD_BDSPR,BEARDO
...,...,...
315,PURSNS_GM,SKIN CARE
316,PURSNS_ML,SKIN CARE
317,TO_CL_OTG,SKIN CARE
318,TO_CL_OTM,SKIN CARE


In [161]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [162]:
trend_file_df['portfolio'].isna().sum()

0

In [163]:
prophet_file_df[
    ['month_date', 'key', 'run_month']
].duplicated().sum()

0

In [164]:
prophet_file_df

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,yhat_60_%ile,yhat_70_%ile,yhat_75_%ile,trend_60_%ile,...,great_indian_festival_lag_2,great_indian_festival_lag_2_lower,great_indian_festival_lag_2_upper,great_indian_festival_lead_1,great_indian_festival_lead_1_lower,great_indian_festival_lead_1_upper,great_indian_festival_lead_2,great_indian_festival_lead_2_lower,great_indian_festival_lead_2_upper,run_month
0,2023-01-31,0.329761,0.031380,0.567468,0.329761,0.329761,0.349649,0.415352,0.448911,0.329761,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-31
1,2023-02-28,0.316023,0.206513,0.734432,0.316023,0.316023,0.513942,0.575861,0.608197,0.316023,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-31
2,2023-03-31,0.300813,0.103119,0.613063,0.300813,0.300813,0.400214,0.468861,0.500680,0.300813,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-31
3,2023-04-30,0.286094,0.252669,0.798258,0.286094,0.286094,0.580529,0.640779,0.670557,0.286094,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-31
4,2023-05-31,0.270884,0.214632,0.732215,0.270884,0.270884,0.530148,0.583295,0.617651,0.270884,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22078,2026-11-30,31.503627,35.750528,39.590269,31.503627,31.503627,38.058740,38.545969,38.810118,31.503627,...,4.624185,4.624185,4.624185,0.0,0.0,0.0,0.0,0.0,0.0,2026-08-31
22079,2026-12-31,32.965080,32.607062,36.595362,32.965080,32.965080,34.993597,35.470347,35.697896,32.965080,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,2026-08-31
22080,2027-01-31,34.426532,38.151162,41.889707,34.426532,34.426532,40.283662,40.703969,40.923848,34.426532,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,2026-08-31
22081,2027-02-28,35.746554,32.017026,36.105302,35.746553,35.746554,34.418689,34.891353,35.115592,35.746554,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,2026-08-31


In [165]:
# Merge 70th percentile Prophet predictions
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    prophet_file_df[['month_date', 'key', 'run_month', 'yhat_70_%ile','yhat_60_%ile']].rename(
        columns={
            'yhat_70_%ile': 'pred_prophet_70%ile',
            'yhat_60_%ile': 'pred_prophet_60%ile'
        }
    ),
    on=['month_date', 'key', 'run_month'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [166]:
assert trend_file_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0

In [167]:
trend_file_df.drop('vol_in_rum', axis=1, inplace=True)

In [168]:
offtake_df.head()

,month_date,platform_name,parent_material_code,vol_in_rum,indexbpm,brand_code,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,...,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,run_month,key
0,2026-07-31,Amazon ARIPL,718288,33.954,47.150311,SAFF GOLD,0,0,0,0,...,1,0,0,0,0,1,NaN,3,2026-08-31,Amazon ARIPL_718288
1,2026-08-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,0,...,0,0,0,0,1,0,NaN,3,2026-08-31,Amazon ARIPL_718288
2,2026-09-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,1,0,0,...,0,1,0,0,0,0,0.0,3,2026-08-31,Amazon ARIPL_718288
3,2026-10-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,1,0,...,0,0,1,0,0,0,0.0,4,2026-08-31,Amazon ARIPL_718288
4,2026-11-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,1,...,0,0,0,1,0,0,0.0,4,2026-08-31,Amazon ARIPL_718288


In [169]:
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0

In [170]:
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])


In [171]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    offtake_df[['key', 'run_month','month_date', 'vol_in_rum']],
    on=['run_month','month_date', 'key'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [172]:
trend_file_df.select_dtypes('number').isna().sum()

pred_SARIMA                       484
pred_p3m                            0
pred_p6m                            0
pred_prophet                    28347
pred_rf                             0
pred_value_SARIMA                 484
pred_value_p3m                      0
pred_value_p6m                      0
pred_value_prophet              28347
pred_value_rf                       0
parent_material_code                0
great_indian_festival           95255
great_indian_festival_lag_1     95255
great_indian_festival_lag_2     95255
great_indian_festival_lead_1    95255
great_indian_festival_lead_2    95255
ratio_last_year                 10722
quarter                             0
qtr_ind_rate                        0
vol_in_rum_value                    0
vol_in_rum_treated                  0
vol_in_rum_value_treated            0
cov                                 0
big_billion_days                27372
big_billion_days_lag_1          27372
big_billion_days_lag_2          27372
big_billion_

In [173]:
trend_file_df.select_dtypes('number').min().round()

pred_SARIMA                      -7764.0
pred_p3m                             0.0
pred_p6m                             0.0
pred_prophet                         0.0
pred_rf                              0.0
pred_value_SARIMA                   -3.0
pred_value_p3m                       0.0
pred_value_p6m                       0.0
pred_value_prophet                   0.0
pred_value_rf                        0.0
parent_material_code            709538.0
great_indian_festival                0.0
great_indian_festival_lag_1          0.0
great_indian_festival_lag_2          0.0
great_indian_festival_lead_1         0.0
great_indian_festival_lead_2         0.0
ratio_last_year                      0.0
quarter                              1.0
qtr_ind_rate                         0.0
vol_in_rum_value                     0.0
vol_in_rum_treated                   0.0
vol_in_rum_value_treated             0.0
cov                                  0.0
big_billion_days                     0.0
big_billion_days

In [174]:
trend_file_df['vol_in_rum'].fillna(0, inplace=True)

In [175]:
for col in [ 'pred_prophet_70%ile','pred_prophet_60%ile', 'vol_in_rum']:
    trend_file_df[col] = trend_file_df[col].clip(lower=0)

In [176]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,...,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum
0,Amazon RK_710542,2024-09-30,0.000000,0.000330,0.000165,NaN,0.000196,0.000000,0.000012,0.000006,...,NaN,NaN,NaN,NaN,2026-08-31,None,CNO,NaN,NaN,0.00045
1,Amazon RK_710542,2024-10-31,0.000221,0.000330,0.000165,NaN,0.000151,0.000008,0.000012,0.000006,...,NaN,NaN,NaN,NaN,2026-08-31,None,CNO,NaN,NaN,0.00054
2,Amazon RK_710542,2024-11-30,0.000266,0.000330,0.000165,NaN,0.000101,0.000009,0.000012,0.000006,...,NaN,NaN,NaN,NaN,2026-08-31,None,CNO,NaN,NaN,0.00000
3,Amazon RK_710542,2024-12-31,0.000000,0.000330,0.000165,NaN,0.000052,0.000000,0.000012,0.000006,...,NaN,NaN,NaN,NaN,2026-08-31,None,CNO,NaN,NaN,0.00000
4,Amazon RK_710542,2025-01-31,0.000000,0.000180,0.000165,NaN,0.000000,0.000000,0.000006,0.000006,...,NaN,NaN,NaN,NaN,2026-08-31,None,CNO,NaN,NaN,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122622,Purplle_809250,2026-11-30,0.000940,0.000933,0.000917,0.000000,0.001135,0.000272,0.000270,0.000265,...,0.0,1.0,0.0,0.0,2026-08-31,M+3,PREM. HAIR NOUR.,0.000000,0.000000,0.00000
122623,Purplle_809250,2026-12-31,0.001487,0.000933,0.000917,0.000330,0.000798,0.000430,0.000270,0.000265,...,0.0,0.0,0.0,0.0,2026-08-31,M+4,PREM. HAIR NOUR.,0.000651,0.000494,0.00000
122624,Purplle_809250,2027-01-31,0.001487,0.000933,0.000917,0.000199,0.001010,0.000430,0.000270,0.000265,...,0.0,0.0,0.0,0.0,2026-08-31,M+5,PREM. HAIR NOUR.,0.000520,0.000379,0.00000
122625,Purplle_809250,2027-02-28,0.000940,0.000933,0.000917,0.000648,0.001168,0.000272,0.000270,0.000265,...,0.0,0.0,0.0,0.0,2026-08-31,M+6,PREM. HAIR NOUR.,0.000996,0.000834,0.00000


In [177]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [178]:
trend_file_df['P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

trend_file_df['LY P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['LY P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [179]:
trend_file_df['LY P3M_copy'] = trend_file_df['LY P3M'].copy()

In [180]:
trend_file_df['P3M Max'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()

In [181]:
trend_file_df['P3M Top 2 Mean'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False

In [182]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [183]:
trend_file_df['MoM P3M growth'] = (
    trend_file_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)

In [184]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['MoM P3M growth_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
trend_file_df['MoM P3M growth_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)



In [185]:
trend_file_df['>=20%_3M_inc_month_count'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 

In [186]:
trend_file_df['Avg(P3M Mean, Max)'] = trend_file_df[['P3M', 'P3M Max']].mean(axis=1)

In [187]:
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
#         trend_file_df[col] = trend_file_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )

In [188]:
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    trend_file_df[f'{col}_value'] = trend_file_df[col] * trend_file_df['qtr_ind_rate'] / (10 ** 7)

In [189]:
trend_file_df['vol_in_rum_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['vol_in_rum'] / (10 ** 7)
trend_file_df['pred_prophet_70%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_70%ile'] / (10 ** 7)
trend_file_df['pred_prophet_60%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_60%ile'] / (10 ** 7)

In [190]:
value_cols = [col for col in trend_file_df.columns if 'value' in col]
value_cols

['pred_value_SARIMA',
 'pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'vol_in_rum_value',
 'vol_in_rum_value_treated',
 'P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'pred_prophet_70%ile_value',
 'pred_prophet_60%ile_value']

In [191]:
for col in value_cols:
    try:
        assert trend_file_df[col].min() >= 0
    except:
        print(col)
    

    # trend_file_df[col] = trend_file_df[col] / (10 ** 7)

pred_value_SARIMA


In [192]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,...,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value
1617,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,NaN,0.396000,0.000000,0.000025,0.000056,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1618,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,NaN,0.273000,0.000022,0.000025,0.000056,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1619,Amazon RK_718472,2024-01-31,0.143208,0.51,1.125,NaN,0.879000,0.000007,0.000025,0.000056,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1620,Amazon RK_718472,2024-02-29,0.786187,0.51,1.125,NaN,1.275000,0.000039,0.000025,0.000056,...,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,NaN,NaN
1621,Amazon RK_718472,2024-03-31,0.738595,0.90,1.125,NaN,1.998000,0.000037,0.000045,0.000056,...,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34765,Big Basket_719192,2026-11-30,0.000000,0.00,0.000,NaN,0.000000,0.000000,0.000000,0.000000,...,-100.0,-100.0,0.0,0.00,0.000000,0.0,0.0,0.0,NaN,NaN
34766,Big Basket_719192,2026-12-31,0.000000,0.00,0.000,NaN,0.000000,0.000000,0.000000,0.000000,...,-100.0,-100.0,0.0,0.00,0.000000,0.0,0.0,0.0,NaN,NaN
34767,Big Basket_719192,2027-01-31,0.000000,0.00,0.000,NaN,0.000000,0.000000,0.000000,0.000000,...,-100.0,-100.0,0.0,0.00,0.000000,0.0,0.0,0.0,NaN,NaN
34768,Big Basket_719192,2027-02-28,0.000000,0.00,0.000,NaN,0.000000,0.000000,0.000000,0.000000,...,-100.0,-100.0,0.0,0.00,0.000000,0.0,0.0,0.0,NaN,NaN


In [193]:
assert trend_file_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0

In [194]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['LY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

trend_file_df['LLY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


trend_file_df['LY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

trend_file_df['LLY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


trend_file_df['OT_Value_in_Cr_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

trend_file_df['OT_Value_in_Cr_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

trend_file_df['OT_Value_in_Cr_lag_3'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)

In [195]:
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [196]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'brand_code',
       'great_indian_festival', 'great_indian_festival_lag_1',
       'great_indian_festival_lag_2', 'great_indian_festival_lead_1',
       'great_indian_festival_lead_2', 'ratio_last_year', 'quarter',
       'qtr_ind_rate', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'big_billion_days', 'big_billion_days_lag_1',
       'big_billion_days_lag_2', 'big_billion_days_lead_1',
       'big_billion_days_lead_2', 'run_month', 'M month', 'portfolio',
       'pred_prophet_70%ile', 'pred_prophet_60%ile', 'vol_in_rum', 'P3M',
       'P6M', 'LY P3M', 'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean',
       'MoM P3M growth', 'MoM P3M growth_lag_1'

In [197]:
# trend_file_df[['ASM', 'Depot', 'PSKU']] = trend_file_df['key'].str.split('_', expand=True)

In [198]:
trend_file_df.reset_index(drop=True, inplace=True)

In [199]:
trend_file_df.shape

(122627, 67)

In [200]:
trend_file_df['key'].nunique()

2847

In [201]:
# batch_info = pd.read_excel(
#     '/data/aniket/az_demand_forecasting-mil-sc/channel_wise_batch.xlsx'
# )

In [202]:
# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()

In [203]:
# brand_class_df.columns = ['brand_code', 'class']


In [204]:
# len_before_merge = len(trend_file_df)
# trend_file_df = trend_file_df.merge(
#     brand_class_df, 
#     on=['brand_code'],
#     how='left'
# )
# assert len_before_merge == len(trend_file_df)
# del len_before_merge

In [205]:
# trend_file_df['class'].isna().sum()

In [206]:
# trend_file_df['class'].unique()

missing combinations

In [207]:
# model_file = pd.read_csv("/data/aman_singh/acuuracy_check/Heuristic_QCOM_Chain_PSKU_Offtakes_live2.csv")
# model_file

In [208]:
model_file = trend_file_df.copy()

In [209]:
model_file['key'].nunique()

2847

In [210]:
run_month

Timestamp('2026-08-31 00:00:00')

In [211]:
data_query = f"""
    select * from {input_table}
    where month_date >= '2023-01-01' and run_month = '{month_run}'
        
"""
qcom_df = pd.read_sql(data_query, dev_conn)
qcom_df.head()

,MONTH_DATE,PLATFORM_NAME,PARENT_MATERIAL_CODE,VOL_IN_RUM,INDEXBPM,BRAND_CODE,IMPUTED,BIG_BILLION_DAYS,BIG_BILLION_DAYS_LAG_1,BIG_BILLION_DAYS_LAG_2,BIG_BILLION_DAYS_LEAD_1,BIG_BILLION_DAYS_LEAD_2,GREAT_INDIAN_FESTIVAL,GREAT_INDIAN_FESTIVAL_LAG_1,GREAT_INDIAN_FESTIVAL_LAG_2,GREAT_INDIAN_FESTIVAL_LEAD_1,GREAT_INDIAN_FESTIVAL_LEAD_2,RATIO_LAST_YEAR,QUARTER,RUN_MONTH
0,2026-07-31,Amazon ARIPL,718288,33.954,47.150311,SAFF GOLD,0,0,0,0,0,1,0,0,0,0,1,NaN,3,2026-08-31
1,2026-08-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,0,1,0,0,0,0,1,0,NaN,3,2026-08-31
2,2026-09-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,1,0,0,0,0,1,0,0,0,0,0.0,3,2026-08-31
3,2026-10-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,1,0,0,0,0,1,0,0,0,0.0,4,2026-08-31
4,2026-11-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,1,0,0,0,0,1,0,0,0.0,4,2026-08-31


In [212]:
qcom_df.columns = qcom_df.columns.str.lower()

In [213]:
qcom_df['key'] = qcom_df[['platform_name', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [214]:
model_file['run_month'] = pd.to_datetime(model_file['run_month'])
model_file['month_date'] = pd.to_datetime(model_file['month_date'])

qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month_date'] = pd.to_datetime(qcom_df['month_date'])

In [215]:
tmp_df = model_file.groupby(['key', 'run_month'])['LY'].count().reset_index()
tmp_df#.isnull().sum()
#qcom_df[~qcom_df['key'].isin(model_file['key'].unique())]
qcom_df = qcom_df.merge(tmp_df, on = ['key', 'run_month'], how = 'left')
missing_df = qcom_df[qcom_df['LY'].isna()]
missing_df


,month_date,platform_name,parent_material_code,vol_in_rum,indexbpm,brand_code,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,...,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,run_month,key,LY
0,2026-07-31,Amazon ARIPL,718288,33.954,47.150311,SAFF GOLD,0,0,0,0,...,0,0,0,0,1,NaN,3,2026-08-31,Amazon ARIPL_718288,NaN
1,2026-08-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,0,...,0,0,0,1,0,NaN,3,2026-08-31,Amazon ARIPL_718288,NaN
2,2026-09-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,1,0,0,...,1,0,0,0,0,0.0,3,2026-08-31,Amazon ARIPL_718288,NaN
3,2026-10-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,1,0,...,0,1,0,0,0,0.0,4,2026-08-31,Amazon ARIPL_718288,NaN
4,2026-11-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,1,...,0,0,1,0,0,0.0,4,2026-08-31,Amazon ARIPL_718288,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138925,2027-01-31,Purplle,810674,0.000,0.000000,PA_ESS_HO,1,0,0,0,...,0,0,0,0,0,NaN,1,2026-08-31,Purplle_810674,NaN
138926,2027-02-28,Purplle,810674,0.000,0.000000,PA_ESS_HO,1,0,0,0,...,0,0,0,0,0,NaN,1,2026-08-31,Purplle_810674,NaN
138927,2027-03-31,Purplle,810674,0.000,0.000000,PA_ESS_HO,1,0,0,0,...,0,0,0,0,0,NaN,1,2026-08-31,Purplle_810674,NaN
138928,2027-04-30,Purplle,810674,0.000,0.000000,PA_ESS_HO,1,0,0,0,...,0,0,0,0,0,0.0,2,2026-08-31,Purplle_810674,NaN


In [216]:
missing_df['key'].nunique()

690

In [217]:
missing_df = missing_df[['key','run_month','month_date', 'platform_name', 'parent_material_code', 'brand_code',
       'vol_in_rum']]
missing_df

,key,run_month,month_date,platform_name,parent_material_code,brand_code,vol_in_rum
0,Amazon ARIPL_718288,2026-08-31,2026-07-31,Amazon ARIPL,718288,SAFF GOLD,33.954
1,Amazon ARIPL_718288,2026-08-31,2026-08-31,Amazon ARIPL,718288,SAFF GOLD,0.000
2,Amazon ARIPL_718288,2026-08-31,2026-09-30,Amazon ARIPL,718288,SAFF GOLD,0.000
3,Amazon ARIPL_718288,2026-08-31,2026-10-31,Amazon ARIPL,718288,SAFF GOLD,0.000
4,Amazon ARIPL_718288,2026-08-31,2026-11-30,Amazon ARIPL,718288,SAFF GOLD,0.000
...,...,...,...,...,...,...,...
138925,Purplle_810674,2026-08-31,2027-01-31,Purplle,810674,PA_ESS_HO,0.000
138926,Purplle_810674,2026-08-31,2027-02-28,Purplle,810674,PA_ESS_HO,0.000
138927,Purplle_810674,2026-08-31,2027-03-31,Purplle,810674,PA_ESS_HO,0.000
138928,Purplle_810674,2026-08-31,2027-04-30,Purplle,810674,PA_ESS_HO,0.000


In [218]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


Credentials retrieved successfully for prod db.


,month_date,brand_code,qtr_ind_rate
0,2027-03-31,CMX_WELPD,3014.000
1,2027-03-31,4700_BCPC,222.000
2,2027-03-31,CMX_PRTPD,3014.000
3,2027-03-31,PA_CN_HGO,488.152
4,2027-03-31,TRU_RAWDF,800.000


In [219]:
len_before_merge = len(missing_df)

missing_df = missing_df.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(missing_df)

In [220]:
missing_df

,key,run_month,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,qtr_ind_rate
0,Amazon ARIPL_718288,2026-08-31,2026-07-31,Amazon ARIPL,718288,SAFF GOLD,33.954,138865.260689
1,Amazon ARIPL_718288,2026-08-31,2026-08-31,Amazon ARIPL,718288,SAFF GOLD,0.000,138865.260689
2,Amazon ARIPL_718288,2026-08-31,2026-09-30,Amazon ARIPL,718288,SAFF GOLD,0.000,138865.260689
3,Amazon ARIPL_718288,2026-08-31,2026-10-31,Amazon ARIPL,718288,SAFF GOLD,0.000,138865.260689
4,Amazon ARIPL_718288,2026-08-31,2026-11-30,Amazon ARIPL,718288,SAFF GOLD,0.000,138865.260689
...,...,...,...,...,...,...,...,...
10604,Purplle_810674,2026-08-31,2027-01-31,Purplle,810674,PA_ESS_HO,0.000,12860.631072
10605,Purplle_810674,2026-08-31,2027-02-28,Purplle,810674,PA_ESS_HO,0.000,12860.631072
10606,Purplle_810674,2026-08-31,2027-03-31,Purplle,810674,PA_ESS_HO,0.000,12860.631072
10607,Purplle_810674,2026-08-31,2027-04-30,Purplle,810674,PA_ESS_HO,0.000,12860.631072


In [221]:
missing_df['month_date'] = pd.to_datetime(missing_df['month_date'])
missing_df['run_month'] = pd.to_datetime(missing_df['run_month'])


mappings = {}

for run_month in missing_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 11):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   
missing_df['M month'] = missing_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)


len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(missing_df)

# Merge 70th percentile Prophet predictions

assert missing_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0
missing_df.drop('vol_in_rum', axis=1, inplace=True)

In [222]:
missing_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7', 'M+8',
       'M+9'], dtype=object)

In [223]:
offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    offtake_df[['key', 'month_date', 'vol_in_rum']],
    on=['month_date', 'key'],
    how='left'
)
assert len(missing_df) == len_before_merge

missing_df['vol_in_rum'].fillna(0, inplace=True)
for col in [ 'vol_in_rum']:
    missing_df[col] = missing_df[col].clip(lower=0)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [224]:
missing_df['P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=1).mean()

missing_df['P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=3).mean()

missing_df['LY P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

missing_df['LY P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [225]:

missing_df['LY P3M_copy'] = missing_df['LY P3M'].copy()
missing_df['P3M Max'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()
missing_df['P3M Top 2 Mean'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth'] = (
    missing_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
missing_df['MoM P3M growth_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)


missing_df['>=20%_3M_inc_month_count'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 
missing_df['Avg(P3M Mean, Max)'] = missing_df[['P3M', 'P3M Max']].mean(axis=1)
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
#         missing_df[col] = missing_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )
# missing_df.to_csv('collate_check.csv', index=False)
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    missing_df[f'{col}_value'] = missing_df[col] * missing_df['qtr_ind_rate'] / (10 ** 7)
missing_df['vol_in_rum_value'] = missing_df['qtr_ind_rate'] * missing_df['vol_in_rum'] / (10 ** 7)
value_cols = [col for col in missing_df.columns if 'value' in col]
value_cols
for col in value_cols:
    try:
        assert missing_df[col].min() >= 0
    except:
        print(col)
    

    # missing_df[col] = missing_df[col] / (10 ** 7)
missing_df
assert missing_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['LY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

missing_df['LLY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


missing_df['LY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

missing_df['LLY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


missing_df['OT_Value_in_Cr_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

missing_df['OT_Value_in_Cr_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

missing_df['OT_Value_in_Cr_lag_3'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

missing_df.reset_index(drop=True, inplace=True)

# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()
# brand_class_df.columns = ['brand_code', 'class']

# len_before_merge = len(missing_df)
# missing_df = missing_df.merge(
#     brand_class_df, 
#     on=['brand_code'],
#     how='left'
# )
# assert len_before_merge == len(missing_df)
# del len_before_merge
# missing_df['class'].isna().sum()
# missing_df['class'].unique()

LY P3M_value


In [226]:
pd.set_option('display.max_columns', None)

In [227]:
# missing_df[
#     # (missing_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (missing_df['month_date'] > '2024-06-30') &
#     (missing_df['M month'].notna())
#     # (missing_df['class'].isin(['B', 'C']))
# ].to_csv('missing_combinations_QCOM_Chain_city_PSKU_Offtakes.csv', index=False)

In [228]:
model_file

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,NaN,0.396000,0.000000,0.000025,0.000056,NaN,1.967441e-05,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,0.000000,4,496.828458,0.000022,0.45,0.000022,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,NaN,0.273000,0.000022,0.000025,0.000056,NaN,1.356342e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,NaN,4,496.828458,0.000000,0.00,0.000000,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000022,NaN,NaN
2,Amazon RK_718472,2024-01-31,0.143208,0.51,1.125,NaN,0.879000,0.000007,0.000025,0.000056,NaN,4.367122e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.400000,1,496.828458,0.000054,1.08,0.000054,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN
3,Amazon RK_718472,2024-02-29,0.786187,0.51,1.125,NaN,1.275000,0.000039,0.000025,0.000056,NaN,6.334563e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,7.200000,1,496.828458,0.000080,1.62,0.000080,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022
4,Amazon RK_718472,2024-03-31,0.738595,0.90,1.125,NaN,1.998000,0.000037,0.000045,0.000056,NaN,9.926633e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,5.294118,1,496.828458,0.000134,2.70,0.000134,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122622,Big Basket_719192,2026-11-30,0.000000,0.00,0.000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000e+00,719192,Big Basket,VEG_CLEAN,NaN,NaN,NaN,NaN,NaN,NaN,4,100.000000,0.000000,0.00,0.000000,2026-07-31,4.724640,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-08-31,M+3,HEALTH & HYGIENE,NaN,NaN,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-100.000000,-100.0,-100.0,0.0,0.00,0.000000,0.0,0.

In [229]:
model_file['skipped'] = 0
missing_df['skipped'] = 1
final_df = pd.concat([model_file,missing_df])
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,NaN,0.396,0.000000,0.000025,0.000056,NaN,0.000020,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,0.000000,4.0,496.828458,0.000022,0.45,0.000022,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,NaN,0.273,0.000022,0.000025,0.000056,NaN,0.000014,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,NaN,4.0,496.828458,0.000000,0.00,0.000000,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000022,NaN,NaN,0
2,Amazon RK_718472,2024-01-31,0.143208,0.51,1.125,NaN,0.879,0.000007,0.000025,0.000056,NaN,0.000044,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.400000,1.0,496.828458,0.000054,1.08,0.000054,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN,0
3,Amazon RK_718472,2024-02-29,0.786187,0.51,1.125,NaN,1.275,0.000039,0.000025,0.000056,NaN,0.000063,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,7.200000,1.0,496.828458,0.000080,1.62,0.000080,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,1.62,0.510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.510,0.000025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022,0
4,Amazon RK_718472,2024-03-31,0.738595,0.90,1.125,NaN,1.998,0.000037,0.000045,0.000056,NaN,0.000099,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,5.294118,1.0,496.828458,0.000134,2.70,0.000134,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,2.70,0.900,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.900,0.000045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10604,Myntra_811169,2027-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,811169,Myntra,SW_SGPRF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1712.605337,0.000000,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-31,M+5,MALE GROOMING,NaN,NaN,0.00,4.896,2.9376,NaN,NaN,NaN,5.22,3.762,-6.206897,126.5625,inf,NaN,5.058,0.000838,0.000503,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000728,0.000999,0.000789,1
10605,Myntra_811169,2027-02-28,NaN,NaN,NaN,NaN,NaN,N

In [230]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.0,0.396,0.000000,0.000025,0.000056,0.0,0.000020,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,0.000000,4.0,496.828458,0.000022,0.45,0.000022,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,0.45,0.000,0.0000,0.0,0.0,0.0,0.00,0.000,0.000000,0.0000,0.0,0.0,0.000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.0,0.273,0.000022,0.000025,0.000056,0.0,0.000014,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,0.000000,4.0,496.828458,0.000000,0.00,0.000000,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,0.00,0.000,0.0000,0.0,0.0,0.0,0.00,0.000,0.000000,0.0000,0.0,0.0,0.000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.000022,0.000000,0.000000,0
2,Amazon RK_718472,2024-01-31,0.143208,0.51,1.125,0.0,0.879,0.000007,0.000025,0.000056,0.0,0.000044,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.400000,1.0,496.828458,0.000054,1.08,0.000054,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,1.08,0.000,0.0000,0.0,0.0,0.0,0.00,0.000,0.000000,0.0000,0.0,0.0,0.000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.000000,0.000022,0.000000,0
3,Amazon RK_718472,2024-02-29,0.786187,0.51,1.125,0.0,1.275,0.000039,0.000025,0.000056,0.0,0.000063,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,7.200000,1.0,496.828458,0.000080,1.62,0.000080,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,1.62,0.510,0.0000,0.0,0.0,0.0,0.00,0.000,0.000000,0.0000,0.0,0.0,0.510,0.000025,0.000000,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.000054,0.000000,0.000022,0
4,Amazon RK_718472,2024-03-31,0.738595,0.90,1.125,0.0,1.998,0.000037,0.000045,0.000056,0.0,0.000099,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,5.294118,1.0,496.828458,0.000134,2.70,0.000134,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,2.70,0.900,0.0000,0.0,0.0,0.0,0.00,0.000,76.470588,0.0000,0.0,0.0,0.900,0.000045,0.000000,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.000080,0.000054,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10604,Myntra_811169,2027-01-31,0.000000,0.00,0.000,0.0,0.000,0.000000,0.000000,0.000000,0.0,0.000000,811169,Myntra,SW_SGPRF,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1712.605337,0.000000,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-08-3

In [231]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.0,0.396,0.000000,0.000025,0.000056,0.0,0.000020,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,0.000000,4.0,496.828458,0.000022,0.45,0.000022,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,0.45,0.000,0.0000,0.0,0.0,0.0,0.00,0.000,0.000000,0.0000,0.0,0.0,0.000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.0,0.273,0.000022,0.000025,0.000056,0.0,0.000014,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,0.000000,4.0,496.828458,0.000000,0.00,0.000000,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,0.00,0.000,0.0000,0.0,0.0,0.0,0.00,0.000,0.000000,0.0000,0.0,0.0,0.000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.000022,0.000000,0.000000,0
2,Amazon RK_718472,2024-01-31,0.143208,0.51,1.125,0.0,0.879,0.000007,0.000025,0.000056,0.0,0.000044,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.400000,1.0,496.828458,0.000054,1.08,0.000054,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,1.08,0.000,0.0000,0.0,0.0,0.0,0.00,0.000,0.000000,0.0000,0.0,0.0,0.000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.000000,0.000022,0.000000,0
3,Amazon RK_718472,2024-02-29,0.786187,0.51,1.125,0.0,1.275,0.000039,0.000025,0.000056,0.0,0.000063,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,7.200000,1.0,496.828458,0.000080,1.62,0.000080,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,1.62,0.510,0.0000,0.0,0.0,0.0,0.00,0.000,0.000000,0.0000,0.0,0.0,0.510,0.000025,0.000000,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.000054,0.000000,0.000022,0
4,Amazon RK_718472,2024-03-31,0.738595,0.90,1.125,0.0,1.998,0.000037,0.000045,0.000056,0.0,0.000099,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,5.294118,1.0,496.828458,0.000134,2.70,0.000134,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,2.70,0.900,0.0000,0.0,0.0,0.0,0.00,0.000,76.470588,0.0000,0.0,0.0,0.900,0.000045,0.000000,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.000080,0.000054,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10604,Myntra_811169,2027-01-31,0.000000,0.00,0.000,0.0,0.000,0.000000,0.000000,0.000000,0.0,0.000000,811169,Myntra,SW_SGPRF,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1712.605337,0.000000,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-08-3

Heuristic new approac

In [232]:
# pip install pymannkendall

identify events

In [233]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])

In [234]:
import numpy as np 
import pandas as pd 
EVENT_MONTHS = [9, 10, 11] # Sep, Oct, Nov
def detect_event_months(df_grp):
    df_grp = df_grp.sort_values("month_date").copy()
    run_month = df_grp["run_month"].max()

    # only historical data
    hist = df_grp[df_grp["month_date"] < run_month].copy()

    # initialize
    df_grp["event_month_flag"] = 0
    df_grp["event_uplift_factor"] = 0.0

    if len(hist) < 12:
        df_grp["event_sensitive_flag"] = 0
        return df_grp

    event_sensitive = 0

    # loop year-wise
    for year in hist["month_date"].dt.year.unique():

        year_df = hist[hist["month_date"].dt.year == year]

        for _, row in year_df.iterrows():

            month = row["month_date"].month

            if month not in EVENT_MONTHS:
                continue

            curr_date = row["month_date"]

            # previous 12 months before this month
            prev_12m = hist[
                (hist["month_date"] < curr_date) &
                (hist["month_date"] >= curr_date - pd.DateOffset(months=12)) &
                (~hist["month_date"].dt.month.isin(EVENT_MONTHS))  # 
            ]

            if len(prev_12m) < 6:
                continue

            prev_12m_avg = prev_12m["vol_in_rum_value"].mean()

            if prev_12m_avg <= 0 or np.isnan(prev_12m_avg):
                continue

            uplift = row["vol_in_rum_value"] / prev_12m_avg

            if uplift > 2:
                mask = df_grp["month_date"] == curr_date
                df_grp.loc[mask, "event_month_flag"] = 1
                df_grp.loc[mask, "event_uplift_factor"] = uplift
                event_sensitive = 1

    df_grp["event_sensitive_flag"] = event_sensitive
    return df_grp


In [235]:
final_df = (
    final_df
    .groupby(
        ["platform_name", "parent_material_code", "run_month"],
        group_keys=False
    )
    .apply(detect_event_months)
)


In [236]:
final_df[(final_df['event_month_flag'] == 1) & (final_df['platform_name'] == 'Flipkart National') & ((final_df['parent_material_code'] == 718939))]

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag
81033,Flipkart National_718939,2023-09-30,114.286409,111.709333,86.266000,172.217655,196.157196,1.343365,1.313073,1.014003,2.024310,2.305705,718939,Flipkart National,SAFF ACTV,0.0,0.0,0.0,0.0,0.0,2.617885,3.0,117543.724493,2.186454,186.012000,2.186454,2026-07-31,0.760893,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,1.0,0.0,2026-08-31,None,SAFFOLA OILS,185.414907,177.587810,186.012000,111.709333,86.266000,0.000000,0.000000,0.000000,99.329333,90.655333,12.463589,21.160915,34.787470,2.0,105.519333,1.313073,1.014003,0.000000,0.000000,2.179436,2.087433,0.000,0.000,0.000000,0.000000,1.593188,0.996912,1.349120,0,1,2.251920,1
81034,Flipkart National_718939,2023-10-31,114.286409,135.454667,108.718000,346.147809,294.945936,1.343365,1.592185,1.277912,4.068750,3.466904,718939,Flipkart National,SAFF ACTV,0.0,0.0,0.0,0.0,0.0,2.498109,4.0,117543.724493,4.293919,365.304000,4.293919,2026-07-31,0.760893,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,1.0,0.0,0.0,0.0,0.0,2026-08-31,None,SAFFOLA OILS,359.937243,351.921780,365.304000,135.454667,108.718000,0.000000,0.000000,0.000000,111.709333,105.519333,21.256356,12.463589,21.160915,2.0,123.582000,1.592185,1.277912,0.000000,0.000000,4.230836,4.136620,0.000,0.000,0.000000,0.000000,2.186454,1.593188,0.996912,0,1,4.422486,1
81035,Flipkart National_718939,2023-11-30,114.286409,228.952000,164.140667,174.270022,153.433170,1.343365,2.691187,1.929371,2.048435,1.803511,718939,Flipkart National,SAFF ACTV,0.0,0.0,0.0,0.0,0.0,0.496041,4.0,117543.724493,2.220166,188.880000,2.220166,2026-07-31,0.760893,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,1.0,0.0,0.0,0.0,2026-08-31,None,SAFFOLA OILS,188.968811,182.066443,188.880000,228.952000,164.140667,0.000000,0.000000,0.000000,135.454667,123.582000,69.024815,21.256356,12.463589,2.0,182.203333,2.691187,1.929371,0.000000,0.000000,2.221210,2.140077,0.000,0.000,0.000000,0.000000,4.293919,2.186454,1.593188,0,1,2.286641,1
81045,Flipkart National_718939,2024-09-30,165.354682,115.513333,84.795333,331.008169,273.481988,1.943641,1.357787,0.996716,3.890793,3.214609,718939,Flipkart National,SAFF ACTV,0.0,0.0,0.0,0.0,0.0,1.872679,3.0,117543.724493,3.490061,296.916000,3.490061,2026-07-31,0.760893,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,1.0,0.0,0.0,1.0,0.0,2026-08-31,None,SAFFOLA OILS,345.500793,338.110080,296.916000,115.513333,84.795333,111.709333,86.266000,111.709333,99.206667,85.028667,16.437067,40.022206,31.017309,2.0,107.360000,1.357787,0.996716,1.313073,1.014003,4.061145,3.974272,186.012,0.000,2.186454,0.000000,2.056310,1.234820,0.782230,0,1,3.377611,1
81046,Flipkart National_718939,2024-10-31,293.009718,192.302667,131.576667,259.421680,270.810634,3.444145,2.260397,1.546601,3.049339,3.183209,718939,Flipkart National,SAFF ACTV,0.0,0.0,0.0,0.0,0.0,3.270130,4.0,117543.724493,2.815877,239.560000,2.815877,2026-07-31,0.760893

In [237]:
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["event_month_flag"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()


In [238]:
adj_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_adj": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_adj": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_adj_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_adj_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()

final_df = final_df.merge(
    adj_df,
    on=["platform_name", "parent_material_code", "run_month"],
    how="left"
)

In [239]:
adj_ly_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_adj": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_adj": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_adj_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_adj_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

In [240]:
final_df = final_df.merge(
    adj_ly_df,
    on=["platform_name", "parent_material_code", "run_month"],
    how="left"
)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.0,0.396000,0.000000,0.000025,0.000056,0.0,1.967441e-05,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000022,0.45,0.000022,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.0,0.00,0.0,0.00000,0.0,0.0,0.0,0.0
1,Meesho_718472,2025-12-31,0.000000,0.00,0.000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000e+00,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,0.0,0.81,0.0,0.00004,NaN,NaN,NaN,NaN
2,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.0,0.273000,0.000022,0.000025,0.000056,0.0,1.356342e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,0.0,4.0,496.828458,0.000000,0.00,0.000000,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.0,0.00,0.0,0.00000,0.0,0.0,0.0,0.0
3,Meesho_718472,2026-01-31,0.000000,0.00,0.000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000e+00,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,0.0,0.81,0.0,0.00004,NaN,NaN,NaN,NaN
4,Amazon RK_718472,2024-01-31,0.143208,0.51,1.125,0.0,0.879000,0.000007,0.000025,0.000056,0.0,4.367122e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.4,1.0,496.828458,0.000054,1.08,0.000054,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.0,0.00,0.0,0.00000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133231,Big Basket_719192,2026-11-30,0.000000,0.00,0.000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000e+00

In [241]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(final_df)

final_df = final_df.merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(final_df)


Credentials retrieved successfully for prod db.


In [242]:
import pandas as pd
import numpy as np
import pymannkendall as mk

def detect_trend_for_group(df_grp):
    """
    Detect final trend flag and p3m_slope_flag separately.
    Must contain 'month_date', 'vol_in_rum', 'run_month'
    """

    # ---------- 1. Sort ----------
    df_grp = df_grp.sort_values("month_date")

    # ---------- 2. Identify run_month ----------
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]

    # if no actual data → no trend
    if df_actual.empty or len(df_actual) < 4:
        return pd.Series({"trend_flag": 0, "p3m_slope_flag": 0})

    # ---------- 3. MK Trend ----------
    series = df_actual["vol_in_rum_value"].astype(float)

    try:
        mk_result = mk.original_test(series)
        if mk_result.trend == "increasing":
            mk_trend = 1
        elif mk_result.trend == "decreasing":
            mk_trend = -1
        else:
            mk_trend = 0
    except:
        mk_trend = 0

    # ---------- 4. P3M Slope ----------
    # last 4 months → take last 3 with shift
    #shifted_series = series.shift(1).dropna()

    p3m_values = series.tail(3).values
    #print(p3m_values)

    if len(p3m_values) < 3:
        slope_flag = 0
    else:
        x = np.arange(3)
        slope = np.polyfit(x, p3m_values, 1)[0]
        slope_flag = 1 if slope > 0 else (-1 if slope < 0 else 0)
        

    return pd.Series({
        "trend_flag": mk_trend,
        "p3m_slope_flag": slope_flag
    })


# ---------------------------------------------------------
# APPLY ON ENTIRE DATASET
# ---------------------------------------------------------

# trend_df = final_df.groupby(
#     ["platform_name", "parent_material_code"]
# ).apply(detect_trend_for_group).reset_index()

# trend_df = final_df[final_df['key'] == 'Zepto_721898'].groupby(
#     ["platform_name", "parent_material_code", "run_month"]
# ).apply(detect_trend_for_group).reset_index()
trend_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(detect_trend_for_group).reset_index()

In [243]:
trend_df["final_trend"] = np.where(
    (trend_df["trend_flag"] == 1) & (trend_df["p3m_slope_flag"] == 1), 1,
    np.where(
        (trend_df["trend_flag"] == -1) & (trend_df["p3m_slope_flag"] == -1), -1,
        0
    )
)
trend_df

,platform_name,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend
0,Amazon ARIPL,718288,2026-08-31,0,0,0
1,Amazon ARIPL,718322,2026-08-31,0,0,0
2,Amazon ARIPL,718328,2026-08-31,0,0,0
3,Amazon ARIPL,718330,2026-08-31,0,0,0
4,Amazon ARIPL,718341,2026-08-31,0,0,0
...,...,...,...,...,...,...
3532,Purplle,807725,2026-08-31,-1,0,0
3533,Purplle,809042,2026-08-31,0,0,0
3534,Purplle,809250,2026-08-31,-1,-1,-1
3535,Purplle,810673,2026-08-31,0,1,0


## detect seasonality

In [244]:
from statsmodels.tsa.stattools import acf
import numpy as np
import pandas as pd

def detect_yearly_seasonality(df_grp, threshold=0.3):
    """
    Detects yearly seasonality using ACF at lag=12 only.
    Uses vol_in_rum as the metric.
    """
    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]
    series = df_actual["vol_in_rum"].astype(float).values

    # Need at least 18 points to compare last year vs this year
    if len(series) < 18:
        return 0

    # Compute ACF up to lag-12
    acf_vals = acf(series, nlags=12, fft=False)

    lag12_acf = acf_vals[12]

    # absolute ACF because seasonal correlation can be negative as well
    if abs(lag12_acf) >= threshold:
        return 1
    else:
        return 0
    

seasonality_df = final_df.groupby(
    ["platform_name", "brand_code", 'run_month']
).apply(detect_yearly_seasonality).reset_index(name="seasonality_flag")

seasonality_df



,platform_name,brand_code,run_month,seasonality_flag
0,Amazon ARIPL,CO_SO_VCN,2026-08-31,0
1,Amazon ARIPL,SAF-MUSLI,2026-08-31,0
2,Amazon ARIPL,SAFF ACTV,2026-08-31,0
3,Amazon ARIPL,SAFF GOLD,2026-08-31,0
4,Amazon ARIPL,SAFF KO,2026-08-31,0
...,...,...,...,...
679,Purplle,P_GOHR_SR,2026-08-31,0
680,Purplle,SW HRGEL,2026-08-31,0
681,Purplle,SW NOGAS,2026-08-31,0
682,Purplle,SW STLDEO,2026-08-31,0


In [245]:
# seasonality_df.to_csv('seasonal_ecom.csv')

In [246]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,platform_name,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2026-08-31,0.235752,0.943006,0.471503,0.0
1,Amazon ARIPL,718322,2026-08-31,0.127718,0.510872,0.255436,0.0
2,Amazon ARIPL,718328,2026-08-31,0.051965,0.207858,0.103929,0.0
3,Amazon ARIPL,718330,2026-08-31,0.038111,0.152444,0.076222,0.0
4,Amazon ARIPL,718341,2026-08-31,0.122764,0.491055,0.245528,0.0


In [247]:
trend_df = trend_df.merge(threshold_df, on = ['platform_name', 'parent_material_code', 'run_month'], how = 'left')
trend_df

,platform_name,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2026-08-31,0,0,0,0.235752,0.943006,0.471503,0.000000
1,Amazon ARIPL,718322,2026-08-31,0,0,0,0.127718,0.510872,0.255436,0.000000
2,Amazon ARIPL,718328,2026-08-31,0,0,0,0.051965,0.207858,0.103929,0.000000
3,Amazon ARIPL,718330,2026-08-31,0,0,0,0.038111,0.152444,0.076222,0.000000
4,Amazon ARIPL,718341,2026-08-31,0,0,0,0.122764,0.491055,0.245528,0.000000
...,...,...,...,...,...,...,...,...,...,...
3532,Purplle,807725,2026-08-31,-1,0,0,0.000000,0.000000,0.000000,0.000000
3533,Purplle,809042,2026-08-31,0,0,0,0.000000,0.000000,0.000000,0.000000
3534,Purplle,809250,2026-08-31,-1,-1,-1,0.000000,0.000659,0.000186,0.000158
3535,Purplle,810673,2026-08-31,0,1,0,0.000018,0.000280,0.000123,0.000052


In [248]:
# trend_df.to_csv('t_s_t_df_ecom.csv')

In [249]:
final_df = final_df.merge(seasonality_df, on = ["platform_name", "brand_code", 'run_month'], how = 'left')
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.0,0.396000,0.000000,0.000025,0.000056,0.0,1.967441e-05,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000022,0.45,0.000022,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.0,0.00,0.0,0.00000,0.0,0.0,0.0,0.0,496.828458,0
1,Meesho_718472,2025-12-31,0.000000,0.00,0.000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000e+00,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,0.0,0.81,0.0,0.00004,NaN,NaN,NaN,NaN,496.828458,0
2,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.0,0.273000,0.000022,0.000025,0.000056,0.0,1.356342e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,0.0,4.0,496.828458,0.000000,0.00,0.000000,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.0,0.00,0.0,0.00000,0.0,0.0,0.0,0.0,496.828458,0
3,Meesho_718472,2026-01-31,0.000000,0.00,0.000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000e+00,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,0.0,0.81,0.0,0.00004,NaN,NaN,NaN,NaN,496.828458,0
4,Amazon RK_718472,2024-01-31,0.143208,0.51,1.125,0.0,0.879000,0.000007,0.000025,0.000056,0.0,4.367122e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.4,1.0,496.828458,0.000054,1.08,0.000054,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.0,0.00,0.0,0.00000,0.0,0.0,0.0,0.0,496.828458,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13323

In [250]:
trend_df.columns

Index(['platform_name', 'parent_material_code', 'run_month', 'trend_flag',
       'p3m_slope_flag', 'final_trend', 'lower_threshold', 'upper_threshold',
       'mean_value', 'std_value'],
      dtype='object')

In [251]:
final_df = final_df.merge(trend_df[['platform_name', 'parent_material_code', 'run_month',
                                    'final_trend','lower_threshold', 'upper_threshold']], on = ["platform_name", "parent_material_code", 'run_month'], how = 'left')
final_df


,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.0,0.396000,0.000000,0.000025,0.000056,0.0,1.967441e-05,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000022,0.45,0.000022,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.0,0.00,0.0,0.00000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
1,Meesho_718472,2025-12-31,0.000000,0.00,0.000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000e+00,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,0.0,0.81,0.0,0.00004,NaN,NaN,NaN,NaN,496.828458,0,0,0.0,0.002401
2,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.0,0.273000,0.000022,0.000025,0.000056,0.0,1.356342e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,0.0,4.0,496.828458,0.000000,0.00,0.000000,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.0,0.00,0.0,0.00000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
3,Meesho_718472,2026-01-31,0.000000,0.00,0.000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000e+00,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,0.0,0.81,0.0,0.00004,NaN,NaN,NaN,NaN,496.828458,0,0,0.0,0.002401
4,Amazon RK_718472,2024-01-31,0.143208,0.51,1.125,0.0,0.879000,0.000007,0.000025,0.000056,0.0,4.367122e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.4,1.0,496.828458,0.000054,1.08,0.000054,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.0,0.00,0.0,0.00000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [252]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.0,0.396000,0.000000,0.000025,0.000056,0.0,1.967441e-05,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000022,0.45,0.000022,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.0,0.00,0.0,0.00000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
1,Meesho_718472,2025-12-31,0.000000,0.00,0.000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000e+00,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,0.0,0.81,0.0,0.00004,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.002401
2,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.0,0.273000,0.000022,0.000025,0.000056,0.0,1.356342e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,0.0,4.0,496.828458,0.000000,0.00,0.000000,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.0,0.00,0.0,0.00000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
3,Meesho_718472,2026-01-31,0.000000,0.00,0.000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000e+00,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,0.0,0.81,0.0,0.00004,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.002401
4,Amazon RK_718472,2024-01-31,0.143208,0.51,1.125,0.0,0.879000,0.000007,0.000025,0.000056,0.0,4.367122e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.4,1.0,496.828458,0.000054,1.08,0.000054,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-08-31,None,HAIR OILS,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.0,0.00,0.0,0.00000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [253]:
final_df.drop(columns = ['big_billion_days', 'big_billion_days_lag_1', 'big_billion_days_lag_2',
       'big_billion_days_lead_1', 'big_billion_days_lead_2', 'great_indian_festival',
       'great_indian_festival_lag_1', 'great_indian_festival_lag_2',
       'great_indian_festival_lead_1', 'great_indian_festival_lead_2'], inplace = True)

In [254]:
final_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'brand_code',
       'ratio_last_year', 'quarter', 'qtr_ind_rate_x', 'vol_in_rum_value',
       'vol_in_rum_treated', 'vol_in_rum_value_treated', 'train_till', 'cov',
       'run', 'step', 'file_path', 'run_month', 'M month', 'portfolio',
       'pred_prophet_70%ile', 'pred_prophet_60%ile', 'vol_in_rum', 'P3M',
       'P6M', 'LY P3M', 'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean',
       'MoM P3M growth', 'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'pred_prophet_60%ile_value', 'LY', 'LLY',
       'LY value', 'LLY value', 'OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2',
    

In [255]:
missing_df['LY P3M'].sum()

0.0

In [256]:
final_df = final_df.sort_values(['key', 'month_date'])

# base LY


# LY lags
final_df['ly_lag1_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(13)
)

final_df['ly_lag2_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(14)
)

# LY leads
final_df['ly_lead1_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(11)
)

final_df['ly_lead2_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(10)
)


In [257]:
final_df[final_df['month_date'] == '2026-08-31']['P3M_value'].sum()

41.86305999493458

In [258]:
brand_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'brand')
brand_seas.columns = brand_seas.columns.str.lower()
brand_seas.rename(columns={'brand':'brand_code', 'months_num':'month', 'flag':'is_seasonal_month'}, inplace=True)

final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['month'] = final_df['month_date'].dt.month
final_df = final_df.merge(brand_seas, on = ['brand_code', 'month'], how = 'left')
final_df['is_seasonal_month'].fillna(0, inplace=True)

psku_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'psku')
psku_seas.columns = psku_seas.columns.str.lower()
psku_seas.rename(columns={'months_num':'month', 'flag':'is_seasonal_month_psku'}, inplace=True)

final_df = final_df.merge(psku_seas[['parent_material_code', 'month','is_seasonal_month_psku']], on = ['parent_material_code', 'month'], how = 'left')
final_df['is_seasonal_month_psku'].fillna(0, inplace=True)
final_df['final_seasonal_month'] = np.where(
    (final_df['is_seasonal_month'] == 1) | (final_df['is_seasonal_month_psku'] == 1), 1, 0
)



In [259]:
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["final_seasonal_month"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()

adj_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_non_seasonal_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()
adj_df


adj_ly_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_non_seasonal_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

adj_df = adj_df.merge(adj_ly_df, on = ['key', 'run_month'], how = 'left')
#adj_df[adj_df['key'] == 'reliance_b2c_2_haryana_718488']
adj_df

,key,run_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,Amazon ARIPL_718288,2026-08-31,33.954000,33.954000,0.471503,0.471503,NaN,NaN,NaN,NaN
1,Amazon ARIPL_718322,2026-08-31,15.130000,15.130000,0.255436,0.255436,NaN,NaN,NaN,NaN
2,Amazon ARIPL_718328,2026-08-31,8.406000,8.406000,0.103929,0.103929,NaN,NaN,NaN,NaN
3,Amazon ARIPL_718330,2026-08-31,6.165000,6.165000,0.076222,0.076222,NaN,NaN,NaN,NaN
4,Amazon ARIPL_718341,2026-08-31,17.681000,17.681000,0.245528,0.245528,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
3532,Purplle_807725,2026-08-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
3533,Purplle_809042,2026-08-31,0.000000,0.000000,0.000000,0.000000,1.800000,1.80000,0.000066,0.000066
3534,Purplle_809250,2026-08-31,0.000933,0.000917,0.000270,0.000265,0.001433,0.00175,0.000415,0.000506
3535,Purplle_810673,2026-08-31,0.107333,0.095667,0.000138,0.000123,NaN,NaN,NaN,NaN


In [260]:
final_df.shape

(133236, 83)

In [261]:
#adj_df.to_csv('seasonal_p3m_qcom.csv', index=False)
#all[all['month_date'].isin(['2025-11-30','2025-12-31','2026-01-31'])].groupby(['key','run_month','month_date','final_seasonal_month'])['vol_in_rum'].sum().reset_index().to_csv('seasonal_month_check.csv', index=False)
final_df = final_df.merge(
    adj_df,
    on=['key'],
    how="left"
)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month_x,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,Amazon ARIPL_718288,2026-07-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,138865.260689,0.471503,0.0,0.0,NaT,0.0,NaN,NaN,NaN,2026-08-31,None,SAFFOLA OILS,0.0,0.0,33.954,0.000000,0.0000,0.0,0.0,0.000000,0.000000,0.000,0.0,0.0,0.000000,0.0,0.000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.000000,0.0,0.0,1,0,0.0,0,33.954000,33.954,0.471503,0.471503,0.0,0.0,0.0,0.0,138865.260689,0,0,0.235752,0.943006,NaN,NaN,NaN,NaN,7,0.0,NaN,0.0,0,2026-08-31,33.954000,33.954,0.471503,0.471503,NaN,NaN,NaN,NaN
1,Amazon ARIPL_718288,2026-08-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,138865.260689,0.000000,0.0,0.0,NaT,0.0,NaN,NaN,NaN,2026-08-31,M,SAFFOLA OILS,0.0,0.0,0.000,16.977000,6.7908,0.0,0.0,0.000000,0.000000,0.000,inf,0.0,0.000000,0.0,16.977,0.235752,0.094301,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.471503,0.0,0.0,1,0,0.0,0,33.954000,33.954,0.471503,0.471503,0.0,0.0,0.0,0.0,138865.260689,0,0,0.235752,0.943006,NaN,NaN,NaN,NaN,8,0.0,NaN,0.0,0,2026-08-31,33.954000,33.954,0.471503,0.471503,NaN,NaN,NaN,NaN
2,Amazon ARIPL_718288,2026-09-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,138865.260689,0.000000,0.0,0.0,NaT,0.0,NaN,NaN,NaN,2026-08-31,M+1,SAFFOLA OILS,0.0,0.0,0.000,16.977000,6.7908,0.0,0.0,0.000000,0.000000,0.000,inf,0.0,0.000000,0.0,16.977,0.235752,0.094301,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.471503,0.0,0.0,1,0,0.0,0,33.954000,33.954,0.471503,0.471503,0.0,0.0,0.0,0.0,138865.260689,0,0,0.235752,0.943006,NaN,NaN,NaN,NaN,9,0.0,NaN,0.0,0,2026-08-31,33.954000,33.954,0.471503,0.471503,NaN,NaN,NaN,NaN
3,Amazon ARIPL_718288,2026-10-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,138865.260689,0.000000,0.0,0.0,NaT,0.0,NaN,NaN,NaN,2026-08-31,M+2,SAFFOLA OILS,0.0,0.0,0.000,16.977000,6.7908,0.0,0.0,0.000000,0.000000,0.000,inf,0.0,0.000000,0.0,16.977,0.235752,0.094301,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.471503,0.0,0.0,1,0,0.0,0,33.954000,33.954,0.471503,0.471503,0.0,0.0,0.0,0.0,138865.260689,0,0,0.235752,0.943006,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-08-31,33.954000,33.954,0.471503,0.471503,NaN,NaN,NaN,NaN
4,Amazon ARIPL_718288,2026-11-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,138865.260689,0.000000,0.0,0.0,NaT,0.0,NaN,NaN,NaN,2026-08-31,M+3,SAFFOLA OILS,0.0,0.0,0.000,16.977000,6.7908,0.0,0.0,0.000000,0.000000,0.000,inf,0.0,0.000000,0.0,16.977,0.235752,0.094301,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.471503,0.0,0.0,1,0,0.0,0,33.954000,33.954,0.471503,0.471503,0.0,0.0,0.0,0.0,138865.260689,0,0,0.235752,0.943006,NaN,NaN,NaN,NaN,11,0.0,NaN,0.0,0,2026-08-

In [262]:
final_df.rename(columns = {"run_month_x":'run_month'}, inplace = True)

In [263]:
all_brand = final_df.groupby(['brand_code', 'run_month','month_date'])['vol_in_rum_value'].sum().reset_index()
all_brand

,brand_code,run_month,month_date,vol_in_rum_value
0,ADV-AHO-R,2026-08-31,2023-01-31,0.383727
1,ADV-AHO-R,2026-08-31,2023-02-28,0.299053
2,ADV-AHO-R,2026-08-31,2023-03-31,0.283299
3,ADV-AHO-R,2026-08-31,2023-04-30,0.269135
4,ADV-AHO-R,2026-08-31,2023-05-31,0.242577
...,...,...,...,...
5792,VEG_CLEAN,2026-08-31,2026-11-30,0.000000
5793,VEG_CLEAN,2026-08-31,2026-12-31,0.000000
5794,VEG_CLEAN,2026-08-31,2027-01-31,0.000000
5795,VEG_CLEAN,2026-08-31,2027-02-28,0.000000


In [264]:

def detect_month_anomaly(df, brand_code, month_num, mon=9,threshold=0.25, months_window=3):
    """
    Detect if a specific month's vol_in_rum_value is >25% different 
    from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
    Parameters:
    - df: input dataframe with 'month_date', 'vol_in_rum_value'
    - brand_code: filter by this brand code
    - month_num: month to check (6, 7, 8, 9)
    - threshold: 25% difference threshold
    - months_window: number of months before and after to compare
    """
    
    df_brand = df[df['brand_code'] == brand_code].sort_values('month_date').copy()
    
    if df_brand.empty:
        return None
    
    df_brand['year'] = df_brand['month_date'].dt.year
    df_brand['month'] = df_brand['month_date'].dt.month
    
    years = sorted(df_brand['year'].unique())
    current_year = years[-1]
    past_years = [y for y in years if y < current_year][-2:]
    
    anomalies = []
    
    for year in past_years:
        df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
        month_data = df_year[df_year['month'] == month_num]
        if month_data.empty:
            continue
        
        month_value = month_data['vol_in_rum_value'].iloc[0]
        
        past_months = [(mon - i - 1) % 12 for i in range(1, months_window + 1)]
        print(past_months)
        next_months = [(mon + i - 1) % 12 + 1 for i in range(1, months_window + 1)]
        
        past_m = df_year[df_year['month'].isin(past_months)]['vol_in_rum_value']
        next_m = df_year[df_year['month'].isin(next_months)]['vol_in_rum_value']
        
        comparison_values = pd.concat([past_m])
        
        if comparison_values.empty:
            continue
        
        pct_diffs = []
        for comp_value in comparison_values:
            if comp_value != 0:
                pct_diff = (month_value - comp_value) / comp_value
                pct_diffs.append(pct_diff)
        
        if pct_diffs:
            positive_diffs = [p for p in pct_diffs if p > 0]
            negative_diffs = [p for p in pct_diffs if p < 0]
            same_sign = len(positive_diffs) == len(pct_diffs) or len(negative_diffs) == len(pct_diffs)
            is_anomaly = len([p for p in pct_diffs if abs(p) > threshold]) == len(pct_diffs) and same_sign
        else:
            is_anomaly = False

        anomalies.append({
            'brand_code': brand_code,
            'month': month_num,
            'year': year,
            'month_value': month_value,
            'num_months_compared': len(comparison_values),
            'pct_diffs_from_each': pct_diffs,
            'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
            'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
            'is_anomaly': is_anomaly,
            'direction': 'higher' if month_value > comparison_values.mean() else 'lower'
        })
    
    if len(anomalies) == 2:
        pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
        return pd.DataFrame(anomalies), pattern_repeats
    
    return pd.DataFrame(anomalies), False


# Check months 6, 7, 8, 9
brands = all_brand['brand_code'].unique()
results = []

for month in [9,10,11,12]:
    for brand in brands:
        df_result, repeats = detect_month_anomaly(all_brand, brand, month)
        if df_result is not None and not df_result.empty:
            df_result['pattern_repeats'] = repeats
            results.append(df_result)

anomaly_summary = pd.concat(results, ignore_index=True)
# print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]
[7, 6, 5]


In [265]:
final_brands = anomaly_summary[anomaly_summary['pattern_repeats'] == True].drop_duplicates(subset=['brand_code'])[['brand_code', 'direction', 'min_pct_diff', 'max_pct_diff']]
final_brands['month_different'] = 1
final_df = final_df.merge(final_brands[['brand_code', 'month_different']], on = 'brand_code', how = 'left')
final_df['month_different'].fillna(0, inplace=True)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,month_different
0,Amazon ARIPL_718288,2026-07-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,138865.260689,0.471503,0.0,0.0,NaT,0.0,NaN,NaN,NaN,2026-08-31,None,SAFFOLA OILS,0.0,0.0,33.954,0.000000,0.0000,0.0,0.0,0.000000,0.000000,0.000,0.0,0.0,0.000000,0.0,0.000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.000000,0.0,0.0,1,0,0.0,0,33.954000,33.954,0.471503,0.471503,0.0,0.0,0.0,0.0,138865.260689,0,0,0.235752,0.943006,NaN,NaN,NaN,NaN,7,0.0,NaN,0.0,0,2026-08-31,33.954000,33.954,0.471503,0.471503,NaN,NaN,NaN,NaN,1.0
1,Amazon ARIPL_718288,2026-08-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,138865.260689,0.000000,0.0,0.0,NaT,0.0,NaN,NaN,NaN,2026-08-31,M,SAFFOLA OILS,0.0,0.0,0.000,16.977000,6.7908,0.0,0.0,0.000000,0.000000,0.000,inf,0.0,0.000000,0.0,16.977,0.235752,0.094301,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.471503,0.0,0.0,1,0,0.0,0,33.954000,33.954,0.471503,0.471503,0.0,0.0,0.0,0.0,138865.260689,0,0,0.235752,0.943006,NaN,NaN,NaN,NaN,8,0.0,NaN,0.0,0,2026-08-31,33.954000,33.954,0.471503,0.471503,NaN,NaN,NaN,NaN,1.0
2,Amazon ARIPL_718288,2026-09-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,138865.260689,0.000000,0.0,0.0,NaT,0.0,NaN,NaN,NaN,2026-08-31,M+1,SAFFOLA OILS,0.0,0.0,0.000,16.977000,6.7908,0.0,0.0,0.000000,0.000000,0.000,inf,0.0,0.000000,0.0,16.977,0.235752,0.094301,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.471503,0.0,0.0,1,0,0.0,0,33.954000,33.954,0.471503,0.471503,0.0,0.0,0.0,0.0,138865.260689,0,0,0.235752,0.943006,NaN,NaN,NaN,NaN,9,0.0,NaN,0.0,0,2026-08-31,33.954000,33.954,0.471503,0.471503,NaN,NaN,NaN,NaN,1.0
3,Amazon ARIPL_718288,2026-10-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,138865.260689,0.000000,0.0,0.0,NaT,0.0,NaN,NaN,NaN,2026-08-31,M+2,SAFFOLA OILS,0.0,0.0,0.000,16.977000,6.7908,0.0,0.0,0.000000,0.000000,0.000,inf,0.0,0.000000,0.0,16.977,0.235752,0.094301,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.471503,0.0,0.0,1,0,0.0,0,33.954000,33.954,0.471503,0.471503,0.0,0.0,0.0,0.0,138865.260689,0,0,0.235752,0.943006,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-08-31,33.954000,33.954,0.471503,0.471503,NaN,NaN,NaN,NaN,1.0
4,Amazon ARIPL_718288,2026-11-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,138865.260689,0.000000,0.0,0.0,NaT,0.0,NaN,NaN,NaN,2026-08-31,M+3,SAFFOLA OILS,0.0,0.0,0.000,16.977000,6.7908,0.0,0.0,0.000000,0.000000,0.000,inf,0.0,0.000000,0.0,16.977,0.235752,0.094301,0.0,0.0,0.0,0.0,0.000,0.0,0.000000,0.0,0.471503,0.0,0.0,1,0,0.0,0,33.954000,33.954,0.471503,0.471503,0.0,0.0,0.0,0.0,138865.260689,0,0,0.235752,0.943006,NaN,NaN,NaN

In [276]:
final_df[(final_df['M month'].notna())].to_csv('/data/aman_singh/acuuracy_check/all_combination_ecom_aug_pred.csv')

In [277]:
final_df.to_csv('/data/aman_singh/acuuracy_check/all_combination_ecom_trend.csv')

In [280]:
final_df[final_df['month_date'] == '2026-09-30']['pred_value_prophet'].sum()

63.424248649737365

### some checks

In [267]:
query = f"""select * from {input_table}
where run_month = '2026-08-31' """

data = pd.read_sql(con=dev_conn, sql=query)
data.columns = data.columns.str.lower()
data

,month_date,platform_name,parent_material_code,vol_in_rum,indexbpm,brand_code,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,run_month
0,2026-07-31,Amazon ARIPL,718288,33.954,47.150311,SAFF GOLD,0,0,0,0,0,1,0,0,0,0,1,NaN,3,2026-08-31
1,2026-08-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,0,1,0,0,0,0,1,0,NaN,3,2026-08-31
2,2026-09-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,1,0,0,0,0,1,0,0,0,0,0.0,3,2026-08-31
3,2026-10-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,1,0,0,0,0,1,0,0,0,0.0,4,2026-08-31
4,2026-11-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,1,0,0,0,0,1,0,0,0.0,4,2026-08-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138925,2027-01-31,Purplle,810674,0.000,0.000000,PA_ESS_HO,1,0,0,0,0,0,0,0,0,0,0,NaN,1,2026-08-31
138926,2027-02-28,Purplle,810674,0.000,0.000000,PA_ESS_HO,1,0,0,0,0,0,0,0,0,0,0,NaN,1,2026-08-31
138927,2027-03-31,Purplle,810674,0.000,0.000000,PA_ESS_HO,1,0,0,0,0,0,0,0,0,0,0,NaN,1,2026-08-31
138928,2027-04-30,Purplle,810674,0.000,0.000000,PA_ESS_HO,1,0,0,0,0,0,0,0,0,0,0,0.0,2,2026-08-31


In [268]:
data['key'] = (
    data['platform_name'].astype(str) + '_' +
    data['parent_material_code'].astype(str)
)
data = data[data['key'].isin(trend_file_df['key'].unique())]

In [269]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,NaN,0.396000,0.000000,0.000025,0.000056,NaN,1.967441e-05,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,0.000000,4,496.828458,0.000022,0.45,0.000022,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,NaN,0.273000,0.000022,0.000025,0.000056,NaN,1.356342e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,NaN,4,496.828458,0.000000,0.00,0.000000,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000022,NaN,NaN
2,Amazon RK_718472,2024-01-31,0.143208,0.51,1.125,NaN,0.879000,0.000007,0.000025,0.000056,NaN,4.367122e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.400000,1,496.828458,0.000054,1.08,0.000054,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN
3,Amazon RK_718472,2024-02-29,0.786187,0.51,1.125,NaN,1.275000,0.000039,0.000025,0.000056,NaN,6.334563e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,7.200000,1,496.828458,0.000080,1.62,0.000080,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022
4,Amazon RK_718472,2024-03-31,0.738595,0.90,1.125,NaN,1.998000,0.000037,0.000045,0.000056,NaN,9.926633e-05,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,5.294118,1,496.828458,0.000134,2.70,0.000134,2026-07-31,2.658147,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-08-31,None,HAIR OILS,NaN,NaN,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122622,Big Basket_719192,2026-11-30,0.000000,0.00,0.000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000e+00,719192,Big Basket,VEG_CLEAN,NaN,NaN,NaN,NaN,NaN,NaN,4,100.000000,0.000000,0.00,0.000000,2026-07-31,4.724640,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-08-31,M+3,HEALTH & HYGIENE,NaN,NaN,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-100.000000,-100.0,-100.0,0.0,0.00,0.000000,0.0,0.

In [270]:
import pandas as pd

as_of_date = pd.to_datetime("2026-08-31")  # month-end for Feb 2026
data['month_date'] = pd.to_datetime(data['month_date'])
filtered = data[
    
    (data['month_date'] < as_of_date) &
    (data['month_date'] >= as_of_date - pd.DateOffset(months=3))
]


In [271]:
# assert p3m equals
x = filtered.groupby(['month_date'])['vol_in_rum'].sum().reset_index()['vol_in_rum'].mean()
y = trend_file_df[trend_file_df['month_date'] == '2026-08-31']['P3M'].sum()
assert(int(x)==int(y))

In [272]:
(x,y)

(270563.9341092952, 270563.93410929485)

In [273]:
ly_end = as_of_date - pd.DateOffset(years=1)
ly_start = ly_end - pd.DateOffset(months=3)

filtered = data[
    
    (data['month_date'] < ly_end) &
    (data['month_date'] >= ly_start )
]


In [274]:
# p3m ly check may not equal but should be close
x = filtered.groupby(['month_date'])['vol_in_rum'].sum().reset_index()['vol_in_rum'].mean()
y = trend_file_df[trend_file_df['month_date'] == '2026-08-31']['LY P3M'].sum()
(x,y)

(291542.61518391664, 290243.1802519165)

In [275]:
# p3m consistency check
as_of_date = pd.to_datetime('2026-08-31')

next_3_months = pd.date_range(
    start=as_of_date + pd.offsets.MonthEnd(1),
    periods=3,
    freq='M'
)
for dt in next_3_months:
    p3m_sum = trend_file_df.loc[
        trend_file_df['month_date'] == dt, 'P3M'
    ].sum()
    
    print(f"P3M sum for {dt.date()}: {p3m_sum}")


P3M sum for 2026-09-30: 270563.93410929485
P3M sum for 2026-10-31: 270563.93410929485
P3M sum for 2026-11-30: 270563.93410929485


### The end

In [106]:
import numpy as np
import pandas as pd

df = final_df.copy()

# --------------------------------------------------
# 1. SAFE FACTOR FUNCTIONS (NO ERRORS)
# --------------------------------------------------

def safe_div(a, b):
    """Safe division: if error or b<=0 → return 1."""
    try:
        if b is None or b == 0:
            return 1
        return a / b
    except:
        return 1

# recency factor = p3m/p6m (cap at 2)
df["recency_factor"] = df.apply(
    lambda r: min(2, safe_div(r["P3M_value"], r["P6M_value"])),
    axis=1
)

def safe_shrink(r,column_name):
    try:
        ratio = r[column_name] / r["P3M_value"]
        return 1 / np.sqrt(ratio)
    except:
        return 1

df["shrink_ratio_prophet"] = df.apply(lambda r: safe_shrink(r, "pred_value_prophet"), axis=1)
df["shrink_ratio_rf"] = df.apply(lambda r: safe_shrink(r, "pred_value_rf"), axis=1)

# seasonality factor = p3m / p3mLY (cap at 2)
df["seasonality_factor"] = df.apply(
    lambda r: min(2, safe_div(r["P3M_value"], r["LY P3M_value"])),
    axis=1
)


# shrink_ratio = 1 / sqrt(max(forecast/p3m,1)) → safe



# --------------------------------------------------
# 2. HEURISTICS
# --------------------------------------------------

# Recency heuristic: max(recency_factor * P3M, forecast)
df["recency_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M_value"], r["pred_value_prophet"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'], r["pred_value_prophet"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'] * r["recency_factor"],
                  r["pred_value_prophet"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_prophet_value"] = df["pred_value_prophet"] * df["shrink_ratio_prophet"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_prophet_value"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_prophet_value"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_prophet_value"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_prophet_value"]


df["final_heuristic_prophet_value"] = df.apply(apply_final_logic, axis=1)


In [107]:
df["recency_heuristic_rf_value"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M_value"], r["pred_value_rf"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_rf_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'], r["pred_value_rf"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_rf_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'] * r["recency_factor"],
                  r["pred_value_rf"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_rf_value"] = df["pred_value_rf"] * df["shrink_ratio_rf"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_rf_value"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_rf_value"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_rf_value"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_rf_value"]


df["final_heuristic_rf_value"] = df.apply(apply_final_logic, axis=1)


In [108]:
df["error_prophet"] = df["pred_value_prophet"] - df["vol_in_rum_value"]
df["abs_error_prophet"] = df["error_prophet"].abs()

df["error_rf"] = df["pred_value_rf"] - df["vol_in_rum_value"]
df["abs_error_rf"] = df["error_rf"].abs()

df["error_final_heuristic_prophet"] = df["final_heuristic_prophet_value"] - df["vol_in_rum_value"]
df["abs_error_final_heuristic_prophet"] = df["error_final_heuristic_prophet"].abs()

df["error_final_heuristic_rf"] = df["final_heuristic_rf_value"] - df["vol_in_rum_value"]
df["abs_error_final_heuristic_rf"] = df["error_final_heuristic_rf"].abs()


In [109]:
df["recency_heuristic_rf"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M"], r["pred_rf"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_rf"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'], r["pred_rf"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_rf"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'] * r["recency_factor"],
                  r["pred_rf"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_rf"] = df["pred_rf"] * df["shrink_ratio_rf"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_rf"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_rf"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_rf"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_rf"]


df["final_heuristic_rf"] = df.apply(apply_final_logic, axis=1)


In [110]:
df["recency_heuristic_prophet"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M"], r["pred_prophet"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_prophet"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'], r["pred_prophet"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_prophet"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'] * r["recency_factor"],
                  r["pred_prophet"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_prophet"] = df["pred_prophet"] * df["shrink_ratio_prophet"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_prophet"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_prophet"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_prophet"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_prophet"]


df["final_heuristic_prophet"] = df.apply(apply_final_logic, axis=1)


In [112]:
df.to_csv('Heuristics_all_combination_ecom_cp.csv')

In [114]:
df[(df['M month'].notna())].to_csv('Heuristics_all_combination_ecom_cp2.csv')

In [2]:
import pandas as pd
final_df = pd.read_csv('Heuristics_all_combination_ecom_cp.csv')

/tmp/ipykernel_2052077/3783379672.py:2: DtypeWarning: Columns (20,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv('Heuristics_all_combination_ecom_cp.csv')


In [6]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,platform_name,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2025-12-31,0.134916,0.345766,0.219256,0.042170
1,Amazon ARIPL,718321,2025-12-31,0.000000,0.000000,0.000000,0.000000
2,Amazon ARIPL,718322,2025-12-31,0.060035,0.108314,0.079347,0.009656
3,Amazon ARIPL,718323,2025-12-31,0.000000,0.000000,0.000000,0.000000
4,Amazon ARIPL,718328,2025-12-31,0.016588,0.032115,0.022799,0.003105


In [7]:
threshold_df.to_csv('ecom_threshold.csv')

In [162]:
final_df[(final_df['M month'].notna())].to_csv('Heuristics_all_combination_qcom_cp_chk.csv')

In [129]:
final_df[final_df['month_date'] == '2026-02-28']['LY P3M_value'].sum()

123.92211585241307

In [49]:
df['TREND'].unique()

array([ 1,  0, -1])

In [96]:
prophet_output = collate_file('prophet_data_train_till')

downloaded_results\new_pipeline\202508-09_FK_Others_Offtakes\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\new_pipeline\202508-09_FK_Others_Offtakes\train_till_31_Jul_2025\prophet_results\prophet_data_train_till_31_Jul_2025.csv
downloaded_results\new_pipeline\202509_AZ_BB_Offtakes\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\new_pipeline\202509_AZ_BB_Offtakes\train_till_31_Jul_2025\prophet_results\prophet_data_train_till_31_Jul_2025.csv


In [100]:
len_before_merge = len(offtake_df)
offtake_df = offtake_df.merge(
    read_qtr_ind_rate_table()[['brand_code', 'qtr_ind_rate']] ,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(offtake_df)


Credentials retrieved successfully for prod db.


In [ ]:
offtake_df

In [104]:
offtake_df['OT_Value_in_Cr'] = offtake_df['vol_in_rum'] * offtake_df['qtr_ind_rate'] / (10 ** 7)

In [105]:
offtake_df.to_csv('Offtake_realigned_base.csv', index=False)

In [108]:
trend_file_df[trend_file_df['run_month'].isin(['2025-10-31'])].to_csv('Trend_File_Oct_Live_Run_OT.csv', index=False)

In [107]:
trend_file_df[trend_file_df['run_month'].isin(['2025-07-31', '2025-08-31'])].to_csv('Trend_File_SepAug_OT.csv', index=False)

In [97]:
prophet_output.to_csv('ECOM_Prophet_Trend_OT_Chain_PSKU_AugSep.csv', index=False)

In [81]:
forecast_df = trend_file_df[trend_file_df['month_date'] >= trend_file_df['run_month']]

forecast_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,LY P6M,P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,LY,LLY,LY value,LLY value
26,Amazon_718589,2025-03-31,1312.900000,1293.400000,1824.634404,945.661643,0.059090,0.058213,0.082122,0.042562,...,841.200000,0.059090,0.058213,0.027878,0.037860,0.082122,717.3,1428.0,0.032284,0.064271
27,Amazon_718589,2025-04-30,1312.900000,1293.400000,1894.995393,1306.938750,0.059090,0.058213,0.085289,0.058822,...,792.800000,0.059090,0.058213,0.036843,0.035682,0.085289,1053.0,1040.4,0.047393,0.046826
28,Amazon_718589,2025-05-31,1312.900000,1293.400000,2055.487818,1194.591429,0.059090,0.058213,0.092512,0.053765,...,682.250000,0.059090,0.058213,0.039944,0.030706,0.092512,1241.1,1236.3,0.055859,0.055643
29,Amazon_718589,2025-06-30,1312.900000,1293.400000,1713.411024,989.591786,0.059090,0.058213,0.077116,0.044539,...,811.600000,0.059090,0.058213,0.045178,0.036528,0.077116,797.4,1176.0,0.035889,0.052929
56,Amazon_722188,2025-03-31,0.666667,6.400000,0.000000,471.682496,0.000030,0.000288,0.000000,0.021229,...,782.533333,0.000030,0.000288,0.043045,0.035220,0.000000,806.4,236.8,0.036294,0.010658
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120425,Flipkart National_809422,2025-10-31,211.833333,159.550000,262.584032,129.597491,0.030576,0.023029,0.037901,0.018706,...,207.383333,0.030576,0.023029,0.009040,0.029934,0.046132,0.8,NaN,0.000115,NaN
120442,Flipkart National_809423,2025-07-31,75.000000,53.533333,0.000000,169.044957,0.010826,0.007727,0.000000,0.024400,...,NaN,0.010826,0.007727,0.057582,NaN,0.008194,243.4,NaN,0.035132,NaN
120443,Flipkart National_809423,2025-08-31,75.000000,53.533333,0.000000,117.175693,0.010826,0.007727,0.000000,0.016913,...,NaN,0.010826,0.007727,0.054599,NaN,0.000000,99.6,NaN,0.014376,NaN
120444,Flipkart National_809423,2025-09-30,75.000000,53.533333,0.000000,111.920263,0.010826,0.007727,0.000000,0.016155,...,286.050000,0.010826,0.007727,0.048200,0.041288,0.000000,75.4,NaN,0.010883,NaN


In [98]:
trend_file_df[trend_file_df['run_month'] == '2025-08-31'].to_csv('Trend_file_Sep.csv', index=False)

In [82]:
forecast_df['is_na'] = forecast_df['vol_in_rum'].isna()
forecast_df.groupby(['run_month', 'month_date'])['is_na'].sum()

run_month   month_date
2025-03-31  2025-03-31    0
            2025-04-30    0
            2025-05-31    0
            2025-06-30    0
2025-04-30  2025-04-30    0
            2025-05-31    0
            2025-06-30    0
            2025-07-31    0
2025-05-31  2025-05-31    0
            2025-06-30    0
            2025-07-31    0
            2025-08-31    0
2025-06-30  2025-06-30    0
            2025-07-31    0
            2025-08-31    0
            2025-09-30    0
2025-07-31  2025-07-31    0
            2025-08-31    0
            2025-09-30    0
            2025-10-31    0
Name: is_na, dtype: int64

In [83]:
forecast_df.drop('is_na', axis=1, inplace=True)

In [84]:
pred_value_cols = [col for col in trend_file_df.columns if 'value' in col and 'pred' in col]
pred_value_cols

['pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'pred_value_best_model',
 'pred_prophet_70%ile_value']

In [85]:
for col in pred_value_cols:
    trend_file_df[f'error_{col}'] = trend_file_df[col] - trend_file_df['vol_in_rum_value']
    trend_file_df[f'abs_error_{col}'] = np.abs(trend_file_df[col] - trend_file_df['vol_in_rum_value'])    

In [86]:
trend_file_df.to_csv('Trend_file_OT_FK_AZ_BB.csv', index=False)

In [85]:
feature_importance_df = collate_file('feature_importance_train_till')

downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_28_Feb_2025\ml_results\feature_importance_train_till_28_Feb_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_30_Apr_2025\ml_results\feature_importance_train_till_30_Apr_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_30_Jun_2025\ml_results\feature_importance_train_till_30_Jun_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_31_Mar_2025\ml_results\feature_importance_train_till_31_Mar_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_31_May_2025\ml_results\feature_importance_train_till_31_May_2025.csv


In [87]:
feature_importance_df.to_excel('202505-AZ_BB_OT_Feature_Imp.xlsx', index=False)

In [11]:
collate_file('prophet_data_train_till_').to_excel('202504-08_Prophet_File.xlsx', index=False)

downloaded_results\ECOM_ChainPSKU_OT_run\train_till_28_Feb_2025\prophet_results\prophet_data_train_till_28_Feb_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_30_Apr_2025\prophet_results\prophet_data_train_till_30_Apr_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_31_Mar_2025\prophet_results\prophet_data_train_till_31_Mar_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_31_May_2025\prophet_results\prophet_data_train_till_31_May_2025.csv


### Secondary

In [87]:
sec_query = """SELECT 
    chain,
    parent_material_code, 
    month_date, 
    SUM(sec_actuals_vol_rum_month) AS sec_vol_actuals_rum_month,
    SUM(sec_apo_plan_vol_rum_month) AS sec_apo_plan_vol_rum_month
FROM (
    SELECT 
        month_date, 
        distributor_code, 
        material_code, 
        sec_actuals_vol_rum_month, 
        sec_apo_plan_vol_rum_month
    FROM 
        dwh_bpm_dist_brand_mth_sbp 
    WHERE 
        month_date BETWEEN '2022-04-01' AND '2025-12-31'
) A
JOIN (
    SELECT DISTINCT
        customer, 
        chain
    FROM 
        mst_chain_master 
    WHERE 
        chain_type = 'E Com B2C'
) CC
    ON A.distributor_Code = CC.customer
JOIN (
    SELECT 
        material_code, 
        parent_material_code 
    FROM 
        mst_material 
    WHERE 
        company_code = 'MIL' 
        AND latest_record_ind = 1
) M 
    ON A.material_code = M.material_code
GROUP BY 
    chain,
    parent_material_code, 
    month_date
ORDER BY 
    chain,
    parent_material_code, 
    month_date;
"""


results = pd.read_sql(con=prod_conn, sql=sec_query)
sales_data = pd.DataFrame(results)
sales_data.columns = sales_data.columns.str.lower()
sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})

In [88]:
sales_data['month_date'] = pd.to_datetime(sales_data['month_date'])

In [89]:
sales_data['chain'] = sales_data['chain'].replace({
    'Grofers': 'Blinkit',
    'Amazon B2C': 'Amazon ARIPL',
    'Flipkart-Grocery': 'Flipkart Grocery',
    'FlipkartGrocery': 'Flipkart Grocery',
    'Flipkart-National': 'Flipkart National',
    'RK WORLDINFOCOM': 'Amazon RK',
    'ZEPTO': 'Zepto',
    'Kiranakart Technologies': 'Zepto',
    'Big basket B2B': 'Big Basket',
    'Big basket B2C': 'Big Basket',
    'Myntra': 'MYNTRA'
})

In [90]:
chains = ['Big Basket', 'Blinkit', 'Amazon ARIPL', 'Flipkart Grocery',
       'Flipkart National', 'Swiggy', 'Zepto', 'Amazon RK', 'City Mall',
       '1MG', 'Dealshare', 'Meesho', 'Nykaa', 'First Cry', 'MYNTRA',
       'Purplle']

In [91]:
for chain in chains:
    if chain not in sales_data['chain'].unique():
        print(chain)

In [92]:
dev_conn = get_dbconnection('DEV')
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment_2""",
    dev_conn
)

realignment_df.columns = realignment_df.columns.str.lower()

def realign_pskus(data, channel='ECOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "All")
    ]
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']
    
    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data['parent_material_code'] == old_psku, "parent_material_code"
        ] = new_psku

    return data


Credentials retrieved successfully for dev db.


In [93]:
realigned_df = realign_pskus(sales_data.copy())

In [94]:
realigned_df = realigned_df.groupby(
    ['chain', 'parent_material_code', 'month_date'], as_index=False
).sum()

In [95]:
old_pskus = realignment_df[
    (realignment_df["channel"] == "ECOM")
    | (realignment_df["channel"] == "ECOM" + " B2C")
    | (realignment_df["channel"] == "All")
]['psku old'].unique()

for psku in old_pskus:
    assert psku not in realigned_df['parent_material_code'].unique()

In [96]:
realigned_df['key'] = realigned_df['chain'] + '_' + realigned_df['parent_material_code'].astype(str) 
realigned_df['month_date'] = pd.to_datetime(realigned_df['month_date'])

realigned_df.duplicated(subset=['key', 'month_date']).sum()

0

In [97]:
realigned_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

0

In [99]:
realigned_df


,chain,parent_material_code,month_date,sec_vol_actuals_rum_month,sec_apo_plan_vol_rum_month,key
0,1MG,715095,2024-01-31,0.0,0.000,1MG_715095
1,1MG,715096,2023-05-31,0.0,0.000,1MG_715096
2,1MG,715096,2023-06-30,0.0,0.189,1MG_715096
3,1MG,715096,2023-07-31,0.0,0.016,1MG_715096
4,1MG,715096,2023-08-31,0.0,0.000,1MG_715096
...,...,...,...,...,...,...
1081559,imli,807033,2025-04-30,0.0,0.000,imli_807033
1081560,imli,807033,2025-05-31,0.0,0.000,imli_807033
1081561,imli,807033,2025-06-30,0.0,0.000,imli_807033
1081562,imli,807033,2025-07-31,0.0,0.000,imli_807033


In [105]:
trend_file_df['platform_name'].unique()

array(['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'],
      dtype=object)

In [106]:
trend_file_df['platform_updated'] = np.where(
    trend_file_df['platform_name'] == 'Amazon', 
    np.where(
        trend_file_df['portfolio'].isin(['Foods', 'Saffola Oils']), 
        'Amazon ARIPL', 
        'Amazon RK'
    ),
    trend_file_df['platform_name']
)

In [108]:
trend_file_df['platform_updated'].unique()

array(['Amazon RK', 'Big Basket', 'Flipkart Grocery', 'Flipkart National',
       'Amazon ARIPL'], dtype=object)

In [109]:
trend_file_df.groupby(
    'platform_name'
)['platform_updated'].unique()

platform_name
Amazon               [Amazon RK, Amazon ARIPL]
Big Basket                        [Big Basket]
Flipkart Grocery            [Flipkart Grocery]
Flipkart National          [Flipkart National]
Name: platform_updated, dtype: object

In [102]:
trend_file_df['portfolio']

0             Hair Oils
1             Hair Oils
2             Hair Oils
3             Hair Oils
4             Hair Oils
              ...      
120441    Male Grooming
120442    Male Grooming
120443    Male Grooming
120444    Male Grooming
120445    Male Grooming
Name: portfolio, Length: 120446, dtype: object

In [113]:
trend_file_df['key'] = trend_file_df[
    ['platform_updated', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [116]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    realigned_df[['key', 'month_date', 'sec_vol_actuals_rum_month']],
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(trend_file_df)
del len_before_merge